# Paper figures at print size

Every results figure of the bounded-summary paper, redrawn from the saved tables of the other figure notebooks (so this notebook runs after them and after `tables_three_benchmarks`) and from three saved reads under the study roots: w/o memory's 4,000-step Key-to-Door windows, MazeRunner on one map with memory wiped at every attempt, and the attention tables. Nothing is recomputed: the estimates, 95 % joint seed/task intervals and per-seed values are the saved ones.

Every figure is drawn at the ICLR text width (5.5 in) with 7-point type, so it prints at that size with `\includegraphics[width=\linewidth]`. One legend per figure sits above the panels; colours, line styles and markers follow the shared palette (Okabe–Ito, the method vermillion and the thickest line), with markers, hatching and dark bar edges as the second encoding the colour-vision check asks for. Fonts are embedded as TrueType.

- **paper_results_at_a_glance**: the long-running tasks, two rows of square panels: the same task repeated and a new layout or map inside the run (columns), at the end of the run (top: bars with 95 % intervals, the three seeds as dots, RSM w/o memory dashed; the new-layout Key-to-Door bars computed from the saved relayout panels) and over the whole run per 500-step window (bottom: bands; steps beyond the training horizon shaded, the arrivals of a new layout or map ticked).
- **paper_in_context**: adaptation within one task at the training horizon (Key-to-Door success by attempt, CountRecall cumulative accuracy, MazeRunner goals reached, T-Maze success at the trained corridor over all tasks and by cue side, where a policy that ignores the cue turns the same way on every task).
- **paper_countrecall**: RSM's accuracy on each query along the stream with its summary kept, cleared at every boundary and absent (w/o memory), beside every compared memory's accuracy by the age of the evidence (in CountRecall the evidence age is almost the segment index, so the by-age panel replaces a per-query comparison of the memories).
- **paper_attention**: where RSM's decisions read (summary share by position per block) and what reading the summary is worth (the read-bias dose).
- **paper_transplant**: the summary transplant at one boundary (Key-to-Door doors and the donor's key cell, CountRecall answers against the true and the donor-consistent counts).
- **paper_representation_detail** (appendix): what the summary holds, one row per benchmark: the summary one segment reads, projected on principal components fitted on each seed's development captures and coloured by the hidden variable, beside the ridge read-out R² per target and seed (Key-to-Door, CountRecall); over repeated MazeRunner attempts on one map, the centred similarity of the summaries across rewrites (RSM, RSM-R) and their centred norm.
- **paper_ablations**: carrying the summary and the rewrite rule on the long-running tasks.
- **paper_state**: per-agent deployment state against task length (measured peaks at 4,000 steps; linear for the full history and Memo, flat for RSM, the window and the GRU).
- **paper_interventions** (main text, replacing the clearing and blocking tables): the frozen RSM policy with its summary kept, cleared at every boundary, unreadable or with its earlier buffer unreadable, beside w/o memory, on every benchmark and on one MazeRunner map over 4,000 steps, and (f) the transplant: how often the next Key-to-Door attempt visits the donor's key first. **paper_state**, **paper_retention_age** and **paper_room_attention** are drawn at half width to sit beside the text or each other.
- **paper_retention_age**: CountRecall accuracy by the age of the queried evidence, every compared memory (one panel).
- **paper_room_attention** (main text): where in the Key-to-Door room the decisions read the summary: the summary share minus the seed's mean at the decision's segment position, averaged per room cell, attempt 1 against later attempts (training horizon, every confirmation task, seeds pooled).
- **paper_new_tasks**, **paper_learning** (appendix): a new Key-to-Door layout every 500 steps and a new MazeRunner map every 1,000 steps (per 100 steps, per layout or map, steps to the first success; the per-100-step rates and first doors computed from the saved panels); learning at matched experience.

The representation figures read the tables `scripts/representation_read.py read` writes under each root's `reports/<protocol>/confirmation/representation/`; the 4,000-step displays (similarity, norms) read its `first64/` folder, the first 64 confirmation tasks, which the paper reports.

A panel whose saved table is missing says *records pending*; the notes list every file read and every file missing.

In [ ]:
"""Locate the repository, the study roots and the output folders.

Environment overrides, for a copy of the records elsewhere:
  REASONED_ICRL_OUTPUTS   study roots (default <repo>/outputs)
  FIGURE_OUTPUT_ROOT      where the figures are written (default notebooks/outputs/figures);
                          the tables they redraw are read from, and their notes written to,
                          the sibling data/ folder
"""
import os
import sys
from pathlib import Path

from IPython.display import display

REPO = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO / "notebooks"))

from figures_common import default_layout  # noqa: E402

OUTPUTS = Path(os.environ.get("REASONED_ICRL_OUTPUTS", REPO / "outputs"))
FIGURES_DIR = Path(
    os.environ.get("FIGURE_OUTPUT_ROOT", REPO / "notebooks/outputs/figures")
)
DATA_DIR = FIGURES_DIR.parent / "data"
layout = default_layout(REPO, outputs=OUTPUTS)
print(f"records under {OUTPUTS}\nfigures to  {FIGURES_DIR}\ntables from {DATA_DIR}")

In [ ]:
import csv
import math
from collections import defaultdict
from collections.abc import Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
from figures_common import (
    ENDPOINT_CHECKPOINT,
    GRID,
    INK,
    INK_MUTED,
    METHOD_CONDITION,
    SUMMARY_CLEARED,
    BuiltFigure,
    StudyLayout,
    horizon_contract,
    laps_panel,
    style_of,
)
from matplotlib.axes import Axes
from matplotlib.figure import Figure, SubFigure
from matplotlib.legend_handler import HandlerTuple
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, LogLocator, MaxNLocator, NullFormatter
from numpy.typing import NDArray

from reasoned_icrl.analysis.statistics import seed_task_interval
from reasoned_icrl.experiments.horizon import (
    layout_matrix,
    window_door_counts,
    window_goal_counts,
)
from reasoned_icrl.experiments.records import read_benchmark_results, results_file

TEXT_WIDTH = 5.5
"""The ICLR text width in inches. Every figure is drawn at the size it prints,
so its 7-point type prints at 7 points beside the caption; include it at
``width=\\linewidth``."""

COMPARED: tuple[str, ...] = (
    "raw_summary",
    "full_context",
    "full_gru",
    "memo",
    "memo_fixed",
)
"""The method and the four compared memories, in legend and bar order."""
CONTROL = "raw_segment"
ABLATION = "raw_summary_residual"
LABELS: dict[str, str] = {
    "raw_summary": "RSM (ours)",
    "raw_summary_residual": "RSM-R (residual write)",
    "raw_segment": "RSM w/o memory",
    "full_context": "Full-history Transformer",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo, fixed segments",
}
"""Legend names: the row names of the paper's tables."""
BAR_TICKS: dict[str, str] = {
    "raw_summary": "RSM",
    "full_context": "Full\nhistory",
    "full_gru": "GRU",
    "memo": "Memo",
    "memo_fixed": "Memo\nfixed",
}
BUFFER_BLUE = "#0072B2"
"""Okabe-Ito blue: the buffer-read companion, a hue no memory cell uses."""
LAYER_ORANGES: tuple[str, ...] = ("#F4A582", "#D55E00", "#7F2704")
"""RSM's three blocks, light to dark in the method's hue."""
BETAS: tuple[float, ...] = (0.0, 1.0, 2.0, 4.0, 8.0, math.inf)
REWRITES: tuple[float, ...] = (32.5, 64.5, 96.5)
"""CountRecall's summary rewrites fall between queries 32/33, 64/65 and 96/97."""

ENDPOINTS = "tables_three_benchmarks_table1_endpoints.csv"
STATE = "tables_three_benchmarks_table_state_memory_bytes.csv"
KD_ATTEMPTS = "figure2_in_context_adaptation_keydoor_attempts.csv"
CR_CUMULATIVE = "figure2_in_context_adaptation_countrecall_cumulative.csv"
MR_GOALS = "figure2_in_context_adaptation_mazerunner_goals_reached.csv"
TM_SUCCESS = "figure2_in_context_adaptation_tmaze_success.csv"
TM_CUE_SIDE = "figure2_in_context_adaptation_tmaze_cue_side.csv"
KD_WINDOWS = "figure3_beyond_the_training_horizon_keydoor_windows.csv"
CR_RETENTION = "figure3_beyond_the_training_horizon_countrecall_retention.csv"
MAZE_SIZES = "figureA5_beyond_the_training_horizon_all_benchmarks_mazerunner_sizes.csv"
CR_QUERIES = "figure_countrecall_boundary_segments_accuracy.csv"
LAPS_WINDOWS = "figure_mazerunner_repeated_laps_windows.csv"
LAPS_GAIN = "figure_mazerunner_repeated_laps_memory_gain.csv"
MAP_WINDOWS = "figure_mazerunner_map_change_windows.csv"
MAP_PER_MAP = "figure_mazerunner_map_change_per_map.csv"
MAP_FIRST_LAP = "figure_mazerunner_map_change_first_lap.csv"
"""The figure notebooks' saved tables this notebook redraws (it runs after them)."""
KD_H4000 = "reports/dark-key-to-door/confirmation/horizon/h4000/windows.csv"
"""Under the Key-to-Door Memo study: the 4,000-call read, the only saved source
of w/o memory's windows."""
LAPS_WIPED = "reports/mazerunner/confirmation/laps/attempt-cleared/goals_per_window.csv"
KD_RELAYOUT = (
    "reports/dark-key-to-door/confirmation/horizon/h4000-relayout500/windows.csv"
)
KD_RELAYOUT_R = (
    "reports/dark-key-to-door/confirmation/horizon/"
    "h4000-relayout500-raw_summary_residual/windows.csv"
)
"""Under the Key-to-Door Memo study: doors per layout with a new layout every 500 calls."""
"""Under the MazeRunner study: goals per window with memory wiped at every lap."""

Row = dict[str, str]
Value = tuple[float, float, float, list[float]]
"""Estimate, interval lower and upper bound, and the per-seed values."""


def print_rc() -> dict[str, object]:
    """Matplotlib settings for a figure drawn at :data:`TEXT_WIDTH`."""
    return {
        "font.family": "sans-serif",
        "font.sans-serif": ["Liberation Sans", "Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 7.0,
        "axes.labelsize": 7.0,
        "axes.titlesize": 7.0,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "axes.titlepad": 3.0,
        "axes.labelpad": 2.0,
        "legend.fontsize": 6.5,
        "legend.frameon": False,
        "legend.handlelength": 2.2,
        "legend.handletextpad": 0.5,
        "legend.columnspacing": 1.2,
        "legend.borderaxespad": 0.2,
        "legend.labelspacing": 0.3,
        "xtick.labelsize": 6.3,
        "ytick.labelsize": 6.3,
        "xtick.major.size": 2.0,
        "ytick.major.size": 2.0,
        "xtick.minor.size": 1.2,
        "ytick.minor.size": 1.2,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.minor.width": 0.4,
        "ytick.minor.width": 0.4,
        "xtick.major.pad": 1.5,
        "ytick.major.pad": 1.5,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "xtick.labelcolor": INK,
        "ytick.labelcolor": INK,
        "axes.edgecolor": INK_MUTED,
        "axes.linewidth": 0.5,
        "axes.labelcolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "text.color": INK,
        "lines.linewidth": 1.0,
        "lines.markersize": 2.6,
        "lines.solid_capstyle": "round",
        "lines.solid_joinstyle": "round",
        "lines.dash_capstyle": "butt",
        "hatch.linewidth": 0.5,
        "figure.constrained_layout.h_pad": 0.02,
        "figure.constrained_layout.w_pad": 0.02,
        "figure.constrained_layout.hspace": 0.04,
        "figure.constrained_layout.wspace": 0.04,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 150,
    }


# ------------------------------------------------------------------ records


def read_rows(path: Path) -> list[Row]:
    if not path.is_file():
        return []
    with path.open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))


def number(text: str | None) -> float | None:
    if text is None or text.strip() == "":
        return None
    return math.inf if text.strip() in ("inf", "Infinity") else float(text)


@dataclass
class Sources:
    """The saved tables of the figure notebooks (``data``) and the study roots
    (``layout``); every path read or missing is kept for the notes."""

    data: Path
    layout: StudyLayout
    read: list[str] = field(default_factory=list)
    missing: list[str] = field(default_factory=list)

    def table(self, name: str) -> list[Row]:
        return self._rows(self.data / name)

    def study(self, environment: str, condition: str, relative: str) -> list[Row]:
        return self._rows(self.layout.roots[(environment, condition)] / relative)

    def _rows(self, path: Path) -> list[Row]:
        rows = read_rows(path)
        (self.read if rows else self.missing).append(self.layout.relative(path))
        return rows


@dataclass
class Plotted:
    """Every drawn value, one row per point, saved beside the figure."""

    rows: list[dict[str, object]] = field(default_factory=list)

    def add(
        self,
        panel: str,
        condition: str,
        xs: Sequence[float],
        ys: Sequence[float],
        lower: Sequence[float] | None = None,
        upper: Sequence[float] | None = None,
    ) -> None:
        for index, (x, y) in enumerate(zip(xs, ys, strict=True)):
            row: dict[str, object] = {
                "panel": panel,
                "condition": condition,
                "x": x,
                "estimate": y,
            }
            if lower is not None and upper is not None:
                row["lower"], row["upper"] = lower[index], upper[index]
            self.rows.append(row)


def by_condition(rows: Sequence[Row], **match: str) -> dict[str, list[Row]]:
    out: dict[str, list[Row]] = defaultdict(list)
    for row in rows:
        if all(row.get(key) == value for key, value in match.items()):
            out[
                row.get("condition") or row.get("variant") or row.get("cell", "")
            ].append(row)
    return dict(out)


def series(
    rows: Sequence[Row], x: str
) -> tuple[list[float], list[float], list[float], list[float]]:
    ordered = sorted(rows, key=lambda r: float(r[x]))
    return (
        [float(r[x]) for r in ordered],
        [float(r["estimate"]) for r in ordered],
        [float(r["lower"]) for r in ordered],
        [float(r["upper"]) for r in ordered],
    )


def seeds_of(row: Mapping[str, str]) -> list[float]:
    """Per-seed values from ``seed_<s>`` columns or a ``per_seed`` string."""
    found = [float(row[k]) for k in ("seed_42", "seed_100", "seed_2026") if row.get(k)]
    if found:
        return found
    values = []
    for part in row.get("per_seed", "").strip("{}").split(","):
        if ":" in part:
            values.append(float(part.split(":", 1)[1].strip().strip("'\"")))
    return values


def value_of(row: Mapping[str, str], prefix: str = "") -> Value | None:
    estimate = number(row.get(f"{prefix}estimate"))
    lower = number(row.get(f"{prefix}lower"))
    upper = number(row.get(f"{prefix}upper"))
    if estimate is None or lower is None or upper is None:
        return None
    seeds = seeds_of(
        row if not prefix else {"per_seed": row.get(f"{prefix}per_seed", "")}
    )
    return estimate, lower, upper, seeds


def last_window(rows: Sequence[Row]) -> Row | None:
    return max(rows, key=lambda r: int(r["window"])) if rows else None


def with_centres(rows: Sequence[Row]) -> list[Row]:
    """Window rows with the window's centre in calls (500-call windows)."""
    return [
        {**r, "centre": r.get("centre") or str((int(r["window"]) - 0.5) * 500.0)}
        for r in rows
    ]


# ------------------------------------------------------------------ drawing


def line_width(condition: str) -> float:
    return 1.5 if condition == METHOD_CONDITION else 0.95


def layer(condition: str) -> int:
    return 30 if condition == METHOD_CONDITION else style_of(condition).zorder + 10


def curve(
    ax: Axes,
    xs: Sequence[float],
    ys: Sequence[float],
    condition: str,
    *,
    lower: Sequence[float] | None = None,
    upper: Sequence[float] | None = None,
    markers: bool = True,
    markevery: int | None = None,
) -> None:
    """One cell's curve in its paper style, with its interval as a band."""
    st = style_of(condition)
    ax.plot(
        xs,
        ys,
        color=st.color,
        linestyle=st.linestyle,
        linewidth=line_width(condition),
        marker=st.marker if markers else None,
        markersize=2.8 if condition == METHOD_CONDITION else 2.5,
        markerfacecolor="white" if st.open_marker else st.color,
        markeredgecolor=st.color,
        markeredgewidth=0.6,
        markevery=markevery,
        zorder=layer(condition),
    )
    if lower is not None and upper is not None:
        ax.fill_between(
            xs,
            lower,
            upper,
            color=st.color,
            alpha=0.18 if condition == METHOD_CONDITION else 0.10,
            linewidth=0,
            zorder=layer(condition) - 5,
        )


def handle(condition: str, *, label: str | None = None, markers: bool = True) -> Line2D:
    st = style_of(condition)
    return Line2D(
        [],
        [],
        color=st.color,
        linestyle=st.linestyle,
        linewidth=line_width(condition),
        marker=st.marker if markers else None,
        markersize=3.0,
        markerfacecolor="white" if st.open_marker else st.color,
        markeredgecolor=st.color,
        markeredgewidth=0.6,
        label=label or LABELS[condition],
    )


def cleared_handle() -> Line2D:
    return Line2D(
        [],
        [],
        color=SUMMARY_CLEARED.color,
        linestyle=(0, (1, 1.4)),
        linewidth=1.0,
        marker="x",
        markersize=3.0,
        markeredgewidth=0.7,
        label="RSM, summary cleared",
    )


def panel(ax: Axes, letter: str, title: str) -> None:
    ax.set_title(f"({letter}) {title}")


def recessive(ax: Axes, axis: str = "y") -> None:
    ax.grid(axis=axis, color=GRID, linewidth=0.45)
    ax.set_axisbelow(True)


def pending(ax: Axes, text: str = "records pending") -> None:
    ax.text(
        0.5,
        0.5,
        text,
        transform=ax.transAxes,
        ha="center",
        va="center",
        color=INK_MUTED,
        fontsize=6.3,
    )


def mark(ax: Axes, x: float, text: str | None = None, *, y: float = 0.98) -> None:
    """A dotted vertical reference (training horizon, rewrite, map change)."""
    ax.axvline(x, color=INK_MUTED, linewidth=0.6, linestyle=(0, (1, 1.5)), zorder=2)
    if text:
        ax.text(
            x,
            y,
            f" {text}",
            transform=ax.get_xaxis_transform(),
            ha="left",
            va="top",
            fontsize=5.8,
            color=INK_MUTED,
        )


def calls_axis(ax: Axes, *, horizon: bool = True) -> None:
    ax.set_xlim(0, 4000)
    ax.set_xticks([0, 1000, 2000, 3000, 4000])
    ax.set_xticklabels(["0", "1k", "2k", "3k", "4k"])
    ax.set_xlabel("Environment steps")
    if horizon:
        mark(ax, 500.0, "trained")


def top_legend(
    fig: Figure | SubFigure, handles: Sequence[object], *, ncol: int
) -> None:
    fig.legend(handles=list(handles), loc="outside upper center", ncol=ncol)


def bytes_label(value: float, _: object = None) -> str:
    """Decimal units as the paper writes them: 24.6 MB, 3.2 MB, 250 KB, 3.1 KB;
    whole powers of ten (the axis ticks) without decimals."""
    for unit, scale in (("GB", 1e9), ("MB", 1e6), ("KB", 1e3)):
        if value >= scale:
            size = value / scale
            if size == int(size) or size >= 100:
                return f"{size:.0f} {unit}"
            return f"{size:.1f} {unit}"
    return f"{value:.0f} B"


def built(
    name: str, fig: Figure, src: Sources, plotted: Plotted, notes: dict[str, object]
) -> BuiltFigure:
    notes = {**notes, "missing": sorted(set(src.missing))}
    return BuiltFigure(name, fig, {"plotted": plotted.rows}, notes, list(src.read))


# ------------------------------------------------------------ bar summaries


def draw_bars(
    ax: Axes,
    values: Mapping[str, Value],
    reference: float | None,
    fmt: str,
    plotted: Plotted,
    name: str,
    *,
    fraction: bool,
) -> None:
    """One bar per compared memory at a fixed slot, with its 95 % interval,
    its three seeds as dots, its value above, and w/o memory dashed."""
    ax.set_xticks(range(len(COMPARED)))
    ax.set_xlim(-0.6, len(COMPARED) - 0.4)
    ax.tick_params(axis="x", length=0)
    if not values:
        pending(ax)
        return
    highest = max(max([v[2], *v[3]]) for v in values.values())
    if reference is not None:
        highest = max(highest, reference)
    jitter = (-0.16, 0.0, 0.16)
    for slot, condition in enumerate(COMPARED):
        if condition not in values:
            continue
        estimate, lower, upper, seeds = values[condition]
        st = style_of(condition)
        fixed = condition == "memo_fixed"
        ax.bar(
            slot,
            estimate,
            width=0.72,
            color="white" if fixed else st.color,
            edgecolor=st.color if fixed else INK,
            linewidth=0.7 if fixed else 0.35,
            hatch="//////" if fixed else None,
            zorder=3,
        )
        ax.errorbar(
            slot,
            estimate,
            yerr=[[estimate - lower], [upper - estimate]],
            fmt="none",
            ecolor=INK,
            elinewidth=0.6,
            capsize=1.6,
            capthick=0.6,
            zorder=5,
        )
        for offset, seed in zip(jitter, seeds, strict=False):
            ax.plot(
                slot + offset,
                seed,
                marker="o",
                markersize=2.0,
                markerfacecolor="white",
                markeredgecolor=INK,
                markeredgewidth=0.45,
                linestyle="none",
                zorder=6,
            )
        ax.text(
            slot,
            max([upper, *seeds]) + 0.025 * highest,
            fmt.format(estimate),
            ha="center",
            va="bottom",
            fontsize=5.6,
            fontweight="bold" if condition == METHOD_CONDITION else "normal",
            color=INK,
            zorder=7,
            bbox={
                "boxstyle": "square,pad=0.08",
                "facecolor": "white",
                "edgecolor": "none",
            },
        )
        plotted.add(name, condition, [slot], [estimate], [lower], [upper])
    if reference is not None:
        ax.axhline(
            reference,
            color=style_of(CONTROL).color,
            linestyle=(0, (3, 2)),
            linewidth=0.9,
            zorder=2.5,
        )
        plotted.add(name, CONTROL, [-1], [reference])
    ax.set_ylim(0, highest * 1.2)
    if fraction:
        ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
        ax.set_yticklabels(["0", "", "0.5", "", "1"])
    else:
        ax.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
    recessive(ax)


def subtitle(ax: Axes, title: str, detail: str) -> None:
    ax.set_title(title, pad=10.5)
    ax.annotate(
        detail,
        xy=(0, 1),
        xycoords="axes fraction",
        xytext=(0, 2.5),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=6.0,
        color=INK_MUTED,
    )


def later_layout_values(src: Sources) -> tuple[dict[str, Value], float | None]:
    """Doors per later layout (layouts 2-8) of the new-layout Key-to-Door run:
    per task the mean over those layouts, then the joint seed/task interval."""
    values: dict[str, Value] = {}
    reference = None
    for c in (*COMPARED, CONTROL):
        per_seed: dict[int, dict[tuple[int, int], float]] = {}
        for seed in (42, 100, 2026):
            events = keydoor_relayout_panel(src, c, seed)
            if events is None:
                break
            units = sorted(
                {
                    (int(e.task_id), int(e.rollout_seed))
                    for e in events
                    if e.kind == "attempt"
                }
            )
            matrix = layout_matrix(events, units=units, layouts=8)
            per_seed[seed] = {
                u: float(matrix[i, 1:].mean()) for i, u in enumerate(units)
            }
        if len(per_seed) < 3:
            continue
        seeds = sorted(per_seed)
        units = sorted(set.intersection(*(set(per_seed[s]) for s in seeds)))
        grid = np.asarray([[per_seed[s][u] for u in units] for s in seeds])
        estimate, lower, upper = seed_task_interval(grid)
        if c == CONTROL:
            reference = estimate
        else:
            values[c] = (estimate, lower, upper, [float(v) for v in grid.mean(axis=1)])
    return values, reference


def figure_results_at_a_glance(src: Sources) -> BuiltFigure:
    """The long-running tasks: the same task repeated for 4,000 calls and a new
    layout or map inside the run (columns), each as the value at the end of the
    run (top row: bars with 95 % intervals and the three seeds, w/o memory
    dashed) and over the whole run (bottom row: per 500-call window, bands)."""
    plotted = Plotted()
    kd = by_condition(src.table(KD_WINDOWS))
    kd_control = with_centres(
        [
            r
            for r in src.study("keydoor", "memo", KD_H4000)
            if r.get("variant") == CONTROL
        ]
    )
    kd[CONTROL] = kd_control
    laps = by_condition(src.table(LAPS_WINDOWS))
    maps = by_condition(src.table(MAP_PER_MAP))
    relayout = by_condition(
        [
            {**r, "centre": str((int(r["layout"]) - 0.5) * 500.0)}
            for r in src.study("keydoor", "memo", KD_RELAYOUT)
        ]
    )
    map_windows = by_condition(src.table(MAP_WINDOWS))

    def late(
        table: Mapping[str, list[Row]], control: Sequence[Row]
    ) -> tuple[dict[str, Value], float | None]:
        values = {}
        for c in COMPARED:
            row = last_window(table.get(c, []))
            if row is not None and (v := value_of(row)) is not None:
                values[c] = v
        reference = last_window(control)
        return values, float(reference["estimate"]) if reference else None

    def fourth_map() -> tuple[dict[str, Value], float | None]:
        values = {}
        for c in COMPARED:
            row = next((r for r in maps.get(c, []) if r["map"] == "4"), None)
            if row is not None and (v := value_of(row)) is not None:
                values[c] = v
        control = next((r for r in maps.get(CONTROL, []) if r["map"] == "4"), None)
        return values, float(control["estimate"]) if control else None

    bars = (
        ("a", "Key-to-Door", "Doors, last 500 steps", late(kd, kd_control)),
        ("b", "MazeRunner", "Goals, last 500 steps", late(laps, laps.get(CONTROL, []))),
        ("c", "Key-to-Door", "Doors per later layout", later_layout_values(src)),
        ("d", "MazeRunner", "Goals, fourth map", fourth_map()),
    )
    curves = (
        (kd, "e", "Key-to-Door", "Doors per 500 steps", 50, (), ""),
        (laps, "f", "MazeRunner", "Goals per 500 steps", 30, (), ""),
        (
            relayout,
            "g",
            "Key-to-Door",
            "Doors per 500 steps",
            50,
            tuple(500.0 * k for k in range(1, 8)),
            "new layout",
        ),
        (
            map_windows,
            "h",
            "MazeRunner",
            "Goals per 500 steps",
            30,
            (1000.0, 2000.0, 3000.0),
            "new map",
        ),
    )
    cells = (*COMPARED, CONTROL)
    with plt.rc_context(print_rc()):
        fig = plt.figure(figsize=(TEXT_WIDTH, 3.15), layout="constrained")
        left, right = fig.subfigures(1, 2, wspace=0.02)
        left.suptitle("The same task, repeated", fontsize=7.0, fontweight="bold")
        right.suptitle(
            "A new layout or map inside the run", fontsize=7.0, fontweight="bold"
        )
        grids = [left.subplots(2, 2), right.subplots(2, 2)]
        top = [grids[0][0][0], grids[0][0][1], grids[1][0][0], grids[1][0][1]]
        bottom = [grids[0][1][0], grids[0][1][1], grids[1][1][0], grids[1][1][1]]
        for ax, (letter, title, ylabel, (values, reference)) in zip(
            top, bars, strict=True
        ):
            draw_bars(
                ax,
                values,
                reference,
                "{:.1f}",
                plotted,
                f"{letter} {title}",
                fraction=False,
            )
            panel(ax, letter, title)
            ax.set_ylabel(ylabel)
            ax.set_xticklabels([])
            ax.set_box_aspect(1)
        for ax, (table, letter, title, ylabel, ceiling, changes, label) in zip(
            bottom, curves, strict=True
        ):
            # The compared memories first and the method last, so its band
            # and line sit on top; the other bands are kept light.
            for c in (*cells[1:], cells[0]):
                if table.get(c):
                    xs, ys, lo, hi = series(table[c], "centre")
                    band = c != CONTROL
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=lo if band else None,
                        upper=hi if band else None,
                    )
                    plotted.add(f"{letter} {title}", c, xs, ys, lo, hi)
            if not any(table.get(c) for c in cells):
                pending(ax)
            calls_axis(ax, horizon=False)
            if changes:
                arrivals(ax, changes, label)
            else:
                beyond_training(ax)
            ax.set_ylim(0, ceiling * 1.08)
            ax.set_ylabel(ylabel)
            panel(ax, letter, title)
            recessive(ax)
            ax.set_box_aspect(1)
        handles: list[object] = []
        labels: list[str] = []
        for c in COMPARED:
            st = style_of(c)
            fixed = c == "memo_fixed"
            box = Patch(
                facecolor="white" if fixed else st.color,
                edgecolor=st.color if fixed else INK,
                linewidth=0.7 if fixed else 0.35,
                hatch="//////" if fixed else None,
            )
            handles.append((box, handle(c)))
            labels.append(LABELS[c])
        handles.append(handle(CONTROL))
        labels.append(LABELS[CONTROL])
        fig.legend(
            handles=handles,
            labels=labels,
            loc="outside upper center",
            ncol=6,
            handlelength=2.2,
            columnspacing=1.0,
            handler_map={tuple: HandlerTuple(ndivide=1, pad=0.0)},
        )
    return built(
        "paper_results_at_a_glance",
        fig,
        src,
        plotted,
        {
            "figure": "the long-running tasks: the same task repeated and a new layout or map inside the run; top row the value at the end of the run (bars: mean over 256 confirmation tasks x 3 seeds, whiskers 95 % joint seed/task intervals, dots the seeds, dashed line RSM w/o memory), bottom row per 500-call window over the run (bands 95 % joint intervals, RSM strongest)",
        },
    )


# ------------------------------------------------------------ in context


CUE_SIDES: tuple[tuple[str, str, float], ...] = (
    ("up", "^", -0.27),
    ("down", "v", 0.27),
)
"""T-Maze cue side, its marker and its x-offset from the cell's slot."""


def draw_cue_side(
    ax: Axes,
    success: Mapping[str, list[Row]],
    cue: Mapping[str, list[Row]],
    plotted: Plotted,
) -> None:
    """T-Maze success at the trained corridor, one slot per cell: all tasks
    (the cell's marker, 95 % interval) between the up-cue and down-cue success
    (triangles: seed means, the seeds faint behind them). A policy that ignores
    the cue turns the same way on every task, so one side reads 1 and the other
    0; the shaded band is that fixed turn's success over all tasks."""
    cells = (*COMPARED, CONTROL)
    ax.set_xticks(range(len(cells)))
    ax.set_xticklabels(
        [BAR_TICKS.get(c, "w/o memory").replace("\n", " ") for c in cells],
        fontsize=5.6,
        rotation=35,
        ha="right",
        rotation_mode="anchor",
    )
    ax.set_xlim(-0.6, len(cells) - 0.4)
    ax.tick_params(axis="x", length=0)
    if not success or not cue:
        pending(ax)
        return
    first = next(iter(cue.values()))[0]
    up = float(first["tasks_cue_up"])
    up_fraction = up / (up + float(first["tasks_cue_down"]))
    ax.axhspan(
        min(up_fraction, 1 - up_fraction),
        max(up_fraction, 1 - up_fraction),
        color=INK,
        alpha=0.07,
        linewidth=0,
        zorder=1,
    )
    ax.text(
        -0.45,
        max(up_fraction, 1 - up_fraction) + 0.02,
        "fixed turn",
        ha="left",
        va="bottom",
        fontsize=5.6,
        color=INK_MUTED,
    )
    for slot, condition in enumerate(cells):
        st = style_of(condition)
        face = "white" if st.open_marker else st.color
        rows = success.get(condition)
        value = value_of(rows[0]) if rows else None
        if value is not None:
            estimate, lower, upper, _ = value
            ax.errorbar(
                slot,
                estimate,
                yerr=[[estimate - lower], [upper - estimate]],
                color=st.color,
                elinewidth=0.7,
                capsize=0,
                marker=st.marker,
                markersize=3.4,
                markerfacecolor=face,
                markeredgecolor=st.color,
                markeredgewidth=0.6,
                linestyle="none",
                zorder=layer(condition),
            )
            plotted.add("T-Maze", condition, [slot], [estimate], [lower], [upper])
        seeds = cue.get(condition, [])
        for side, marker, dx in CUE_SIDES:
            values = [float(r[f"success_cue_{side}"]) for r in seeds]
            if not values:
                continue
            ax.plot(
                [slot + dx] * len(values),
                values,
                marker=marker,
                markersize=2.4,
                color=st.color,
                alpha=0.6,
                markerfacecolor=face,
                markeredgewidth=0.4,
                linestyle="none",
                zorder=layer(condition) - 1,
            )
            mean = float(np.mean(values))
            ax.plot(
                slot + dx,
                mean,
                marker=marker,
                markersize=3.0,
                color=st.color,
                markerfacecolor=face,
                markeredgewidth=0.6,
                linestyle="none",
                zorder=layer(condition),
            )
            plotted.add(f"T-Maze, {side} cue", condition, [slot + dx], [mean])
    ax.legend(
        handles=[
            Line2D(
                [],
                [],
                marker=m,
                color=INK_MUTED,
                linestyle="none",
                markersize=3.0,
                label=f"{side} cue",
            )
            for side, m, _ in CUE_SIDES
        ],
        loc="lower left",
        fontsize=5.6,
        handlelength=0.8,
        handletextpad=0.3,
        borderaxespad=0.1,
    )
    ax.set_ylim(-0.05, 1.08)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(["0", "0.5", "1"])


def figure_in_context(src: Sources) -> BuiltFigure:
    """Adaptation inside one task at the native horizon."""
    plotted = Plotted()
    attempts = by_condition(
        [r for r in src.table(KD_ATTEMPTS) if r.get("shown") == "True"]
    )
    cumulative = by_condition(src.table(CR_CUMULATIVE))
    goals = by_condition(src.table(MR_GOALS))
    tm_success = by_condition(src.table(TM_SUCCESS))
    tm_cue = by_condition(src.table(TM_CUE_SIDE))
    cells = (*COMPARED, CONTROL)
    banded = (METHOD_CONDITION, CONTROL)
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(
            1,
            4,
            figsize=(TEXT_WIDTH, 1.95),
            layout="constrained",
            gridspec_kw={"width_ratios": (1.0, 1.0, 0.8, 1.35)},
        )
        for ax, table, key, letter, title, markers in (
            (axes[0], attempts, "attempt", "a", "Key-to-Door", True),
            (axes[1], cumulative, "query", "b", "CountRecall", False),
            (axes[2], goals, "goal", "c", "MazeRunner", True),
        ):
            for c in cells:
                if c in table:
                    xs, ys, lo, hi = series(table[c], key)
                    band = c in banded
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=lo if band else None,
                        upper=hi if band else None,
                        markers=markers,
                    )
                    plotted.add(title, c, xs, ys, lo, hi)
            if not table:
                pending(ax)
            panel(ax, letter, title)
            recessive(ax)
        ax = axes[0]
        ax.set_xticks(range(1, 10))
        ax.set_xlim(0.6, 9.4)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Attempt within the task")
        ax.set_ylabel("Door success")
        ax = axes[1]
        for index, boundary in enumerate(REWRITES):
            mark(ax, boundary, "rewrite" if index == 0 else None, y=0.14)
        ax.set_xlim(1, 103)
        ax.set_xticks([1, 32, 64, 96])
        ax.set_ylim(0.6, 1.01)
        ax.set_xlabel("Query")
        ax.set_ylabel("Cumulative exact accuracy")
        ax = axes[2]
        ax.set_xticks([1, 2, 3])
        ax.set_xticklabels(["1st", "2nd", "3rd"])
        ax.set_xlim(0.8, 3.2)
        ax.set_ylim(0.4, 1.0)
        ax.set_xlabel("Goal reached")
        ax.set_ylabel("Fraction of maps")
        ax = axes[3]
        draw_cue_side(ax, tm_success, tm_cue, plotted)
        panel(ax, "d", "T-Maze")
        recessive(ax)
        ax.set_ylabel("Success, corridor 128")
        top_legend(fig, [handle(c) for c in cells], ncol=6)
    return built(
        "paper_in_context",
        fig,
        src,
        plotted,
        {
            "figure": "in-context adaptation at the native horizon; bands for RSM and"
            " w/o memory; T-Maze success over all tasks and by cue side (seed means,"
            " seeds faint), band = a fixed turn's success"
        },
    )


# ------------------------------------------------------------ long running


def beyond_training(ax: Axes, start: float = 500.0) -> None:
    """Shade the calls beyond the training horizon."""
    ax.axvspan(start, 4000, color="#f2f2f2", zorder=0, linewidth=0)


def arrivals(ax: Axes, changes: Sequence[float], text: str) -> None:
    """Mark each new layout or map with a short tick at the top edge."""
    for change in changes:
        ax.plot(
            [change, change],
            [0.94, 1.0],
            transform=ax.get_xaxis_transform(),
            color=INK_MUTED,
            linewidth=0.7,
            zorder=3,
            clip_on=False,
        )
    ax.text(
        changes[0],
        0.93,
        f" {text}",
        transform=ax.get_xaxis_transform(),
        ha="left",
        va="top",
        fontsize=5.6,
        color=INK_MUTED,
    )


# ------------------------------------------------------------ CountRecall


def rewrite_bands(ax: Axes) -> None:
    for boundary in REWRITES:
        ax.axvline(
            boundary, color=INK_MUTED, linewidth=0.6, linestyle=(0, (1, 1.5)), zorder=2
        )


def figure_countrecall(src: Sources) -> BuiltFigure:
    """Evidence that survives the rewrites: RSM with its summary kept, cleared
    or absent, query by query (a), and every compared memory by the age of the
    queried evidence (b)."""
    plotted = Plotted()
    accuracy = src.table(CR_QUERIES)
    retained = by_condition(accuracy, history="retained")
    cleared = by_condition(accuracy, history="summary-cleared")
    retention = by_condition(src.table(CR_RETENTION), history="retained")
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(
            1,
            2,
            figsize=(TEXT_WIDTH, 1.6),
            layout="constrained",
            gridspec_kw={"width_ratios": [1.3, 1.0]},
        )
        ax = axes[0]
        for c in (CONTROL, METHOD_CONDITION):
            if c in retained:
                xs, ys, lo, hi = series(retained[c], "query")
                curve(ax, xs, ys, c, lower=lo, upper=hi, markers=False)
                plotted.add("a", c, xs, ys, lo, hi)
        if METHOD_CONDITION in cleared:
            xs, ys, lo, hi = series(cleared[METHOD_CONDITION], "query")
            ax.plot(
                xs,
                ys,
                color=SUMMARY_CLEARED.color,
                linestyle=(0, (1, 1.4)),
                linewidth=1.0,
                marker="x",
                markersize=2.6,
                markeredgewidth=0.6,
                markevery=(2, 6),
                zorder=25,
            )
            plotted.add("a", SUMMARY_CLEARED.condition, xs, ys, lo, hi)
        if not retained:
            pending(ax)
        rewrite_bands(ax)
        ax.set_xlim(1, 103)
        ax.set_xticks([1, 32, 64, 96])
        ax.set_ylim(-0.02, 1.03)
        ax.set_xlabel("Query along the stream")
        ax.set_ylabel("Exact accuracy")
        panel(ax, "a", "Summary kept, cleared, absent")
        recessive(ax)
        ax = axes[1]
        offsets = dict(
            zip(COMPARED, np.linspace(-0.28, 0.28, len(COMPARED)), strict=True)
        )
        for c in (*COMPARED[1:], COMPARED[0]):
            rows = {int(r["stratum"]): r for r in retention.get(c, [])}
            if not rows:
                continue
            strata = sorted(rows)
            xs = [s + offsets[c] for s in strata]
            ys = [float(rows[s]["estimate"]) for s in strata]
            lo = [float(rows[s]["lower"]) for s in strata]
            hi = [float(rows[s]["upper"]) for s in strata]
            st = style_of(c)
            ax.errorbar(
                xs,
                ys,
                yerr=[np.subtract(ys, lo), np.subtract(hi, ys)],
                color=st.color,
                linestyle=st.linestyle,
                linewidth=line_width(c) * (1.0 if c == METHOD_CONDITION else 0.6),
                marker=st.marker,
                markersize=3.2 if c == METHOD_CONDITION else 2.8,
                markerfacecolor="white" if st.open_marker else st.color,
                markeredgecolor=st.color,
                markeredgewidth=0.6,
                elinewidth=0.8,
                capsize=1.6,
                capthick=0.6,
                zorder=layer(c),
            )
            plotted.add("b", c, strata, ys, lo, hi)
        if not retention:
            pending(ax)
        ax.set_xticks(range(4))
        ax.set_xticklabels(["in buffer", "1", "2", "3"])
        ax.set_xlim(-0.45, 3.45)
        for boundary in (0.5, 1.5, 2.5):
            ax.axvline(boundary, color=GRID, linewidth=0.5, zorder=0)
        lowest = min(
            (float(r["lower"]) for c in COMPARED for r in retention.get(c, [])),
            default=0.6,
        )
        ax.set_ylim(min(0.6, math.floor(lowest * 20) / 20), 1.02)
        ax.set_xlabel("Rewrites since the evidence was dealt")
        ax.set_ylabel("Exact accuracy")
        panel(ax, "b", "Every memory, by evidence age")
        recessive(ax)
        handles = [handle(c, markers=True) for c in COMPARED] + [
            handle(CONTROL, markers=False),
            cleared_handle(),
        ]
        top_legend(fig, handles, ncol=4)
    return built(
        "paper_countrecall",
        fig,
        src,
        plotted,
        {
            "figure": "CountRecall evidence that survives the summary rewrites: RSM per query with its summary kept, cleared and absent; every compared memory by evidence age"
        },
    )


# ------------------------------------------------------------ attention


def attention_rows(src: Sources, environment: str, name: str) -> list[Row]:
    spec = src.layout.spec(environment)
    relative = f"reports/{spec.protocol}/confirmation/attention/{name}.csv"
    return src.study(environment, METHOD_CONDITION, relative)


def draw_profile(ax: Axes, rows: Sequence[Row], plotted: Plotted, name: str) -> None:
    """Summary share of the decisions by position: RSM per block, the
    w/o-memory envelope over its blocks, and the uniform share."""
    if not rows:
        pending(ax)
        return
    means: dict[tuple[str, int, int], list[float]] = defaultdict(list)
    uniform: dict[int, float] = {}
    for r in rows:
        means[(r["cell"], int(r["layer"]), int(r["position"]))].append(
            float(r["summary"])
        )
        uniform[int(r["position"])] = float(r["uniform_summary"])
    positions = sorted(uniform)
    layers = sorted({key[1] for key in means})
    reference = [
        [float(np.mean(means[(CONTROL, layer, p)])) for p in positions]
        for layer in layers
        if (CONTROL, layer, positions[0]) in means
    ]
    if reference:
        ax.fill_between(
            positions,
            np.min(reference, axis=0),
            np.max(reference, axis=0),
            color=style_of(CONTROL).color,
            alpha=0.25,
            linewidth=0,
            zorder=2,
        )
        for index, values in enumerate(reference):
            plotted.add(name, f"{CONTROL} layer {index + 1}", positions, values)
    ax.plot(
        positions,
        [uniform[p] for p in positions],
        color=INK,
        linestyle=(0, (1, 1.2)),
        linewidth=0.8,
        zorder=3,
    )
    for layer in layers:
        if (METHOD_CONDITION, layer, positions[0]) not in means:
            continue
        values = [
            float(np.mean(means[(METHOD_CONDITION, layer, p)])) for p in positions
        ]
        ax.plot(
            positions,
            values,
            color=LAYER_ORANGES[layer % len(LAYER_ORANGES)],
            linewidth=1.3,
            zorder=10 + layer,
        )
        plotted.add(name, f"{METHOD_CONDITION} layer {layer + 1}", positions, values)


def draw_dose(
    ax: Axes, dose: Sequence[Row], readings: Sequence[Row], plotted: Plotted, name: str
) -> float:
    """The primary metric against the read bias on summary or buffer keys."""
    if not dose:
        pending(ax)
        return 1.0
    base = [r for r in dose if r["target"] == "none"]
    top = 0.0
    for target, colour, linestyle, marker in (
        ("summary", style_of(METHOD_CONDITION).color, "-", "o"),
        ("buffer", BUFFER_BLUE, (0, (3, 1.5)), "s"),
    ):
        rows = sorted(
            base + [r for r in dose if r["target"] == target],
            key=lambda r: number(r["beta"]) or 0.0,
        )
        rows = [r for r in rows if number(r["beta"]) in BETAS]
        xs = [BETAS.index(number(r["beta"])) for r in rows]
        ys = [float(r["primary"]) for r in rows]
        lo = [float(r["primary_lower"]) for r in rows]
        hi = [float(r["primary_upper"]) for r in rows]
        ax.errorbar(
            xs,
            ys,
            yerr=[np.subtract(ys, lo), np.subtract(hi, ys)],
            color=colour,
            linestyle=linestyle,
            linewidth=1.2,
            marker=marker,
            markersize=3.0,
            markeredgewidth=0.6,
            markerfacecolor=colour if target == "summary" else "white",
            markeredgecolor=colour,
            elinewidth=0.6,
            capsize=1.3,
            capthick=0.6,
            zorder=10,
        )
        plotted.add(name, f"bias on {target} keys", xs, ys, lo, hi)
        top = max(top, *hi)
    reference = next(
        (
            number(r["estimate"])
            for r in readings
            if r["reading"].startswith("w/o memory")
        ),
        None,
    )
    if reference is not None:
        ax.axhline(
            reference,
            color=style_of(CONTROL).color,
            linestyle=(0, (3, 2)),
            linewidth=0.9,
            zorder=4,
        )
        plotted.add(name, CONTROL, [-1], [reference])
    return top


def profile_handles() -> list[object]:
    return [
        Line2D(
            [], [], color=LAYER_ORANGES[i], linewidth=1.3, label=f"RSM, block {i + 1}"
        )
        for i in range(3)
    ] + [
        Patch(
            facecolor=style_of(CONTROL).color,
            alpha=0.25,
            linewidth=0,
            label="w/o memory, any block",
        ),
        Line2D(
            [],
            [],
            color=INK,
            linestyle=(0, (1, 1.2)),
            linewidth=0.8,
            label="uniform M/(M+i)",
        ),
    ]


def figure_attention(src: Sources) -> BuiltFigure:
    """Where the decisions read (a, b) and what reading the summary is worth
    (c, d), in one row."""
    plotted = Plotted()
    benchmarks = (
        ("keydoor", "Key-to-Door", "Doors in 500 steps"),
        ("countrecall", "CountRecall", "Exact accuracy"),
    )
    with plt.rc_context(print_rc()):
        fig = plt.figure(figsize=(TEXT_WIDTH, 1.95), layout="constrained")
        left, right = fig.subfigures(1, 2, wspace=0.03)
        profiles = left.subplots(1, 2, sharey=True)
        doses = right.subplots(1, 2)
        for index, (environment, title, metric) in enumerate(benchmarks):
            ax = profiles[index]
            draw_profile(
                ax,
                attention_rows(src, environment, "profile"),
                plotted,
                f"{title} profile",
            )
            ax.set_xlim(1, 32)
            ax.set_xticks([1, 16, 32])
            ax.set_ylim(0, 0.85)
            ax.set_xlabel("Position in segment")
            if index == 0:
                ax.set_ylabel("Attention on the summary")
            panel(ax, "ab"[index], title)
            recessive(ax)
            ax = doses[index]
            top = draw_dose(
                ax,
                attention_rows(src, environment, "dose"),
                attention_rows(src, environment, "readings"),
                plotted,
                f"{title} dose",
            )
            ax.set_xticks(range(len(BETAS)))
            ax.set_xticklabels(["0", "1", "2", "4", "8", "\u221e"])
            ax.set_xlim(-0.3, len(BETAS) - 0.7)
            ax.set_ylim(0, top * 1.08)
            ax.set_xlabel("Read bias \u03b2")
            ax.set_ylabel(metric)
            panel(ax, "cd"[index], title)
            recessive(ax)
        top_legend(left, profile_handles(), ncol=3)
        top_legend(
            right,
            [
                Line2D(
                    [],
                    [],
                    color=style_of(METHOD_CONDITION).color,
                    linewidth=1.2,
                    marker="o",
                    markersize=3.0,
                    label="bias on summary keys",
                ),
                Line2D(
                    [],
                    [],
                    color=BUFFER_BLUE,
                    linestyle=(0, (3, 1.5)),
                    linewidth=1.2,
                    marker="s",
                    markersize=3.0,
                    markerfacecolor="white",
                    label="bias on buffer keys",
                ),
                Line2D(
                    [],
                    [],
                    color=style_of(CONTROL).color,
                    linestyle=(0, (3, 2)),
                    linewidth=0.9,
                    label="w/o memory",
                ),
            ],
            ncol=2,
        )
    return built(
        "paper_attention",
        fig,
        src,
        plotted,
        {
            "figure": "attention of RSM's decisions on the summary and the read-bias dose"
        },
    )


# ------------------------------------------------------------ ablations


def figure_ablations(src: Sources) -> BuiltFigure:
    """Carrying the summary and the rewrite rule on the long-running tasks."""
    plotted = Plotted()
    kd = by_condition(src.table(KD_WINDOWS))
    kd[CONTROL] = with_centres(
        [
            r
            for r in src.study("keydoor", "memo", KD_H4000)
            if r.get("variant") == CONTROL
        ]
    )
    laps = by_condition(src.table(LAPS_WINDOWS))
    wiped = with_centres(
        [
            r
            for r in src.study("mazerunner", METHOD_CONDITION, LAPS_WIPED)
            if r.get("cell") == ABLATION
        ]
    )
    maps = by_condition(src.table(MAP_PER_MAP))
    cells = (METHOD_CONDITION, ABLATION, CONTROL)
    wiped_style = {
        "color": style_of(ABLATION).color,
        "linestyle": (0, (1, 1.4)),
        "linewidth": 1.0,
        "marker": "x",
        "markersize": 3.0,
        "markeredgewidth": 0.7,
    }
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(1, 3, figsize=(TEXT_WIDTH, 1.95), layout="constrained")
        for ax, table, key, letter, title, ylabel in (
            (
                axes[0],
                kd,
                "centre",
                "a",
                "Key-to-Door, 4,000 steps",
                "Doors per 500 steps",
            ),
            (
                axes[1],
                laps,
                "centre",
                "b",
                "MazeRunner, same map",
                "Goals per 500 steps",
            ),
            (
                axes[2],
                maps,
                "map",
                "c",
                "New map every 1,000 steps",
                "Goals per 500 steps",
            ),
        ):
            for c in cells:
                if table.get(c):
                    xs, ys, lo, hi = series(table[c], key)
                    band = c != CONTROL
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=lo if band else None,
                        upper=hi if band else None,
                    )
                    plotted.add(title, c, xs, ys, lo, hi)
            if not any(table.get(c) for c in cells):
                pending(ax)
            ax.set_ylabel(ylabel)
            panel(ax, letter, title)
            recessive(ax)
        if wiped:
            xs, ys, lo, hi = series(wiped, "centre")
            axes[1].plot(xs, ys, zorder=25, **wiped_style)
            plotted.add(
                "MazeRunner same map",
                f"{ABLATION} memory wiped each attempt",
                xs,
                ys,
                lo,
                hi,
            )
        for ax in axes[:2]:
            calls_axis(ax)
        axes[0].set_ylim(0, 50)
        axes[1].set_ylim(0, 30)
        ax = axes[2]
        ax.set_xticks([1, 2, 3, 4])
        ax.set_xlim(0.8, 4.2)
        ax.set_ylim(0, 30)
        ax.set_xlabel("Map within the task")
        top_legend(
            fig,
            [handle(c) for c in cells[:2]]
            + [
                Line2D(
                    [], [], label="RSM-R, memory wiped every attempt", **wiped_style
                ),
                handle(CONTROL),
            ],
            ncol=4,
        )
    return built(
        "paper_ablations",
        fig,
        src,
        plotted,
        {
            "figure": "carrying the summary and replacing versus adding on the long-running tasks"
        },
    )


# ------------------------------------------------------------ new tasks (appendix)

OOD_CELLS: tuple[str, ...] = (*COMPARED, ABLATION, CONTROL)
FINE_WINDOW = 100


_RELAYOUT: dict[tuple[Path, str, int], list[Any] | None] = {}


def keydoor_relayout_panel(src: Sources, condition: str, seed: int) -> list[Any] | None:
    """One seed's 4,000-call Key-to-Door panel with a new layout every 500 calls
    (read once per study root and kept for the other figures)."""
    key = (src.layout.run_directory("keydoor", condition, seed), condition, seed)
    if key not in _RELAYOUT:
        _RELAYOUT[key] = _read_keydoor_relayout(src, condition, seed)
    return _RELAYOUT[key]


def _read_keydoor_relayout(src: Sources, condition: str, seed: int) -> list[Any] | None:
    layout = src.layout
    path = (
        layout.run_directory("keydoor", condition, seed)
        / "eval"
        / "confirmation-retained-endpoint-h4000-relayout500"
        / "benchmark_results.json"
    )
    if results_file(path) is None:
        return None
    runs, events = read_benchmark_results(
        path, [horizon_contract(layout.contract("keydoor"), 4000)]
    )
    (run,) = runs
    if run.layout_period != 500 or run.checkpoint != ENDPOINT_CHECKPOINT:
        return None
    src.read.append(layout.relative(path))
    return list(events)


def mazerunner_relayout_panel(
    src: Sources, condition: str, seed: int
) -> list[Any] | None:
    """One seed's MazeRunner laps panel with a new map every 1,000 calls."""
    return laps_panel(
        src.layout,
        condition,
        seed,
        "confirmation",
        "retained",
        4000,
        contract=src.layout.contract("mazerunner"),
        sources=src.read,
        layout_period=1000,
    )


def interval_rows(
    per_seed: Mapping[int, Mapping[tuple[int, int], Sequence[float]]],
) -> list[tuple[int, float, float, float]]:
    """Per column of the per-unit vectors: the mean and the joint seed/task
    interval over the units every seed has."""
    seeds = sorted(per_seed)
    units = sorted(set.intersection(*(set(per_seed[s]) for s in seeds)))
    if not units:
        return []
    stacked = np.asarray(
        [[per_seed[s][u] for u in units] for s in seeds], dtype=np.float64
    )
    out = []
    for column in range(stacked.shape[2]):
        estimate, lower, upper = seed_task_interval(stacked[:, :, column])
        out.append((column, estimate, lower, upper))
    return out


def first_door_calls(
    events: Sequence[Any], *, layouts: int = 8, period: int = 500
) -> dict[tuple[int, int], list[float]]:
    """Calls from a layout's first attempt to its first completed door, per
    unit and layout, censored at the layout period when no door comes."""
    starts: dict[tuple[int, int, int], int] = {}
    doors: dict[tuple[int, int, int], int] = {}
    for event in events:
        if event.kind != "attempt" or event.layout_index is None:
            continue
        key = (int(event.task_id), int(event.rollout_seed), int(event.layout_index))
        starts[key] = min(starts.get(key, 10**9), int(event.start_step))
        if (
            event.numerator
            and event.complete is not False
            and event.end_step is not None
        ):
            doors[key] = min(doors.get(key, 10**9), int(event.end_step))
    units = sorted({(task, rollout) for task, rollout, _ in starts})
    out: dict[tuple[int, int], list[float]] = {}
    for unit in units:
        row = []
        for index in range(layouts):
            key = (*unit, index)
            if key not in starts:
                row.append(float(period))
            elif key in doors:
                row.append(float(min(doors[key] - starts[key] + 1, period)))
            else:
                row.append(float(period))
        out[unit] = row
    return out


def figure_new_tasks(src: Sources) -> BuiltFigure:
    """Appendix: a new Key-to-Door layout every 500 calls (top) and a new
    MazeRunner map every 1,000 calls (bottom): the rate per 100 calls, the
    rate on each layout or map, and the calls to the first success on each."""
    plotted = Plotted()
    per_layout = by_condition(src.study("keydoor", "memo", KD_RELAYOUT))
    per_layout.update(by_condition(src.study("keydoor", "memo", KD_RELAYOUT_R)))
    per_map = by_condition(src.table(MAP_PER_MAP))
    first_lap = by_condition(src.table(MAP_FIRST_LAP))
    fine: dict[str, dict[str, list[tuple[int, float, float, float]]]] = {
        "keydoor": {},
        "mazerunner": {},
    }
    first_door: dict[str, list[tuple[int, float, float, float]]] = {}
    for c in OOD_CELLS:
        kd_windows: dict[int, dict[tuple[int, int], Sequence[float]]] = {}
        kd_first: dict[int, dict[tuple[int, int], Sequence[float]]] = {}
        mr_windows: dict[int, dict[tuple[int, int], Sequence[float]]] = {}
        for seed in (42, 100, 2026):
            events = keydoor_relayout_panel(src, c, seed)
            if events is not None:
                kd_windows[seed] = window_door_counts(
                    events, outer_length=4000, window=FINE_WINDOW
                )
                kd_first[seed] = first_door_calls(events)
            events = mazerunner_relayout_panel(src, c, seed)
            if events is not None:
                mr_windows[seed] = window_goal_counts(
                    events, outer_length=4000, window=FINE_WINDOW
                )
        if len(kd_windows) == 3:
            fine["keydoor"][c] = interval_rows(kd_windows)
            first_door[c] = interval_rows(kd_first)
        if len(mr_windows) == 3:
            fine["mazerunner"][c] = interval_rows(mr_windows)
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(2, 3, figsize=(TEXT_WIDTH, 3.4), layout="constrained")
        rows = (
            (
                "keydoor",
                "Key-to-Door",
                "Doors",
                500.0,
                8,
                per_layout,
                "layout",
                "Layout",
                "Steps to the first door",
            ),
            (
                "mazerunner",
                "MazeRunner",
                "Goals",
                1000.0,
                4,
                per_map,
                "map",
                "Map",
                "Steps to finish attempt 1",
            ),
        )
        for row, (
            key,
            name,
            unit,
            period,
            count,
            table,
            column,
            label,
            first_label,
        ) in enumerate(rows):
            ax = axes[row][0]
            for c in OOD_CELLS:
                found = fine[key].get(c)
                if not found:
                    continue
                xs = [(w + 0.5) * FINE_WINDOW for w, _, _, _ in found]
                ys = [e for _, e, _, _ in found]
                band = c == METHOD_CONDITION
                curve(
                    ax,
                    xs,
                    ys,
                    c,
                    markers=False,
                    lower=[lo for _, _, lo, _ in found] if band else None,
                    upper=[hi for _, _, _, hi in found] if band else None,
                )
                plotted.add(f"{name} per 100 calls", c, xs, ys)
            if not fine[key]:
                pending(ax)
            calls_axis(ax, horizon=False)
            for k in range(1, int(4000 / period)):
                ax.axvline(
                    k * period,
                    color=INK_MUTED,
                    linewidth=0.5,
                    linestyle=(0, (1, 2)),
                    zorder=1,
                )
            ax.set_ylim(bottom=0)
            ax.set_ylabel(f"{unit} per 100 steps")
            panel(ax, "ad"[row], f"{name}, over the task")
            recessive(ax)
            ax = axes[row][1]
            for c in OOD_CELLS:
                if table.get(c):
                    xs, ys, lo, hi = series(table[c], column)
                    band = c == METHOD_CONDITION
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=lo if band else None,
                        upper=hi if band else None,
                    )
                    plotted.add(f"{name} per {column}", c, xs, ys, lo, hi)
            if not table:
                pending(ax)
            ax.set_xticks(range(1, count + 1))
            ax.set_xlim(0.7, count + 0.3)
            ax.set_ylim(0, 50 if key == "keydoor" else 30)
            ax.set_xlabel(label)
            ax.set_ylabel(f"{unit} per 500 steps")
            panel(ax, "be"[row], f"{name}, each {column}")
            recessive(ax)
            ax = axes[row][2]
            if key == "keydoor":
                for c in OOD_CELLS:
                    found = first_door.get(c)
                    if not found:
                        continue
                    xs = [w + 1 for w, _, _, _ in found]
                    ys = [e for _, e, _, _ in found]
                    band = c == METHOD_CONDITION
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=[lo for _, _, lo, _ in found] if band else None,
                        upper=[hi for _, _, _, hi in found] if band else None,
                    )
                    plotted.add("Key-to-Door first door", c, xs, ys)
                drawn = bool(first_door)
            else:
                for c in OOD_CELLS:
                    if first_lap.get(c):
                        xs, ys, lo, hi = series(first_lap[c], "map")
                        band = c == METHOD_CONDITION
                        curve(
                            ax,
                            xs,
                            ys,
                            c,
                            lower=lo if band else None,
                            upper=hi if band else None,
                        )
                        plotted.add("MazeRunner first attempt", c, xs, ys, lo, hi)
                drawn = bool(first_lap)
            if not drawn:
                pending(ax)
            ax.axhline(500, color=INK_MUTED, linewidth=0.6, linestyle=(0, (1, 1.5)))
            ax.set_xticks(range(1, count + 1))
            ax.set_xlim(0.7, count + 0.3)
            ax.set_ylim(0, 560)
            ax.set_xlabel(label)
            ax.set_ylabel(first_label)
            panel(ax, "cf"[row], f"{name}, first success")
            recessive(ax)
        top_legend(fig, [handle(c) for c in OOD_CELLS], ncol=4)
    return built(
        "paper_new_tasks",
        fig,
        src,
        plotted,
        {
            "figure": "a new Key-to-Door layout every 500 calls and a new MazeRunner map every 1,000 calls; per-100-call rates from the saved panels; first door censored at 500 calls; band RSM's 95 % joint seed/task interval",
        },
    )


# ------------------------------------------------------------ learning

LEARNING = "figureA1_learning_matched_experience_curves.csv"
LEARNING_CELLS: tuple[str, ...] = (
    "raw_summary",
    "raw_summary_residual",
    "raw_segment",
    "full_context",
    "full_gru",
    "memo",
    "memo_fixed",
)
LEARNING_PANELS = (
    ("keydoor", "Key-to-Door", 20.0, (0.0, 40.0)),
    ("countrecall", "CountRecall", 0.9, (0.0, 1.0)),
    ("mazerunner", "MazeRunner", 0.5, (0.0, 1.0)),
    ("tmaze", "T-Maze", 1.0, (0.0, 1.05)),
)
"""Column, title, the declared development threshold and its y range."""


def figure_learning(src: Sources) -> BuiltFigure:
    """Appendix: the collection return (top) and the greedy development
    primary (bottom) against charged training calls, every benchmark and
    cell; seed means, the method's seed range shaded."""
    plotted = Plotted()
    rows = src.table(LEARNING)
    curves: dict[tuple[str, str, str], list[Row]] = defaultdict(list)
    for r in rows:
        curves[(r["environment"], r["series"], r["condition"])].append(r)
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(
            2, 4, figsize=(TEXT_WIDTH, 2.9), layout="constrained", sharex=True
        )
        for column, (environment, title, threshold, limits) in enumerate(
            LEARNING_PANELS
        ):
            for row, series in enumerate(("collection", "development")):
                ax = axes[row][column]
                drawn = False
                for c in LEARNING_CELLS:
                    found = sorted(
                        curves.get((environment, series, c), []),
                        key=lambda r: float(r["calls_millions"]),
                    )
                    if not found:
                        continue
                    xs = [float(r["calls_millions"]) for r in found]
                    ys = [float(r["mean"]) for r in found]
                    band = c == METHOD_CONDITION
                    curve(
                        ax,
                        xs,
                        ys,
                        c,
                        lower=[float(r["low"]) for r in found] if band else None,
                        upper=[float(r["high"]) for r in found] if band else None,
                        markers=False,
                    )
                    plotted.add(f"{environment} {series}", c, xs, ys)
                    drawn = True
                if not drawn:
                    pending(ax)
                ax.set_xlim(0, 8)
                ax.set_xticks([0, 4, 8])
                if series == "development":
                    ax.axhline(
                        threshold,
                        color=INK_MUTED,
                        linewidth=0.6,
                        linestyle=(0, (1, 1.5)),
                        zorder=2,
                    )
                    ax.set_ylim(*limits)
                    ax.set_xlabel("Training steps (M)")
                elif environment in ("countrecall", "tmaze"):
                    ax.set_ylim(-1, 1)
                if row == 0:
                    panel(ax, "abcd"[column], title)
                recessive(ax)
        axes[0][0].set_ylabel("Collection return")
        axes[1][0].set_ylabel("Development primary")
        top_legend(fig, [handle(c, markers=False) for c in LEARNING_CELLS], ncol=4)
    return built(
        "paper_learning",
        fig,
        src,
        plotted,
        {
            "figure": "learning at matched experience; smoothed for drawing as in figureA1; RSM's seed range shaded; dotted the declared development thresholds"
        },
    )


# ------------------------------------------------------------ representation

TRANSPLANT_BLUE = "#0072B2"
"""The transplanted summary: Okabe-Ito blue, a hue no memory cell uses."""
SEGMENT_CALLS = 32


FIRST64 = "first64"
"""The 4,000-step representation reads on the first 64 confirmation tasks, the
ones the paper reports."""
FIRST64_TABLES = frozenset({"similarity", "norms"})


def representation_rows(src: Sources, environment: str, name: str) -> list[Row]:
    spec = src.layout.spec(environment)
    folder = f"reports/{spec.protocol}/confirmation/representation"
    if name in FIRST64_TABLES:
        folder = f"{folder}/{FIRST64}"
    return src.study(environment, METHOD_CONDITION, f"{folder}/{name}.csv")


def draw_projection(
    fig: Figure,
    ax: Axes,
    rows: Sequence[Row],
    colour: str,
    label: str,
    plotted: Plotted,
    name: str,
    *,
    seed: str = "42",
) -> None:
    chosen = [r for r in rows if r["seed"] == seed]
    if not chosen:
        pending(ax)
        ax.set_xticks([])
        ax.set_yticks([])
        return
    xs = [float(r["pc1"]) for r in chosen]
    ys = [float(r["pc2"]) for r in chosen]
    values = [float(r[colour]) for r in chosen]
    points = ax.scatter(
        xs,
        ys,
        c=values,
        cmap="viridis",
        s=5,
        linewidths=0,
        vmin=min(values),
        vmax=max(values),
        rasterized=True,
    )
    bar = fig.colorbar(points, ax=ax, shrink=0.85, pad=0.02, aspect=18)
    bar.set_label(label, fontsize=6.3, labelpad=1.5)
    bar.ax.tick_params(labelsize=5.8, length=1.5, width=0.4)
    bar.outline.set_linewidth(0.4)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(f"PC1 ({float(chosen[0]['pc1_share']):.0%})")
    ax.set_ylabel(f"PC2 ({float(chosen[0]['pc2_share']):.0%})")
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(GRID)
    plotted.add(name, METHOD_CONDITION, xs, ys)


def similarity_matrix(
    rows: Sequence[Row], cell: str
) -> tuple[NDArray[np.float64], int] | None:
    chosen = [r for r in rows if r["cell"] == cell]
    if not chosen:
        return None
    segments = sorted({int(r["segment"]) for r in chosen})
    index = {segment: k for k, segment in enumerate(segments)}
    matrix = np.full((len(segments), len(segments)), np.nan)
    for r in chosen:
        matrix[index[int(r["segment"])], index[int(r["other"])]] = float(
            r["similarity"]
        )
    return matrix, segments[0]


def draw_similarity(ax: Axes, rows: Sequence[Row], cell: str) -> Any:
    found = similarity_matrix(rows, cell)
    if found is None:
        pending(ax)
        ax.set_xticks([])
        ax.set_yticks([])
        return None
    matrix, first = found
    last_call = (first + len(matrix)) * SEGMENT_CALLS
    image = ax.imshow(
        matrix,
        cmap="RdBu_r",
        vmin=-1.0,
        vmax=1.0,
        origin="lower",
        extent=(first * SEGMENT_CALLS, last_call, first * SEGMENT_CALLS, last_call),
        interpolation="nearest",
        rasterized=True,
    )
    ticks = [1000, 2000, 3000, 4000]
    ax.set_xticks(ticks)
    ax.set_xticklabels(["1k", "2k", "3k", "4k"])
    ax.set_yticks(ticks)
    ax.set_yticklabels(["1k", "2k", "3k", "4k"])
    ax.set_xlabel("Steps")
    for spine in ax.spines.values():
        spine.set_visible(False)
    return image


def draw_grid(ax: Axes, rows: Sequence[Row], attempts: str, limit: float) -> Any:
    chosen = [r for r in rows if r["attempts"] == attempts]
    if not chosen:
        pending(ax)
        ax.set_xticks([])
        ax.set_yticks([])
        return None
    grid = np.full((8, 8), np.nan)
    for r in chosen:
        if r["share"] != "":
            grid[int(r["row"]), int(r["column"])] = float(r["share"])
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#eeeeee")
    image = ax.imshow(
        np.ma.masked_invalid(grid),
        cmap=cmap,
        vmin=-limit,
        vmax=limit,
        interpolation="nearest",
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    return image


def figure_room_attention(src: Sources) -> BuiltFigure:
    """Where in the Key-to-Door room the decisions read the summary: the
    position-adjusted summary share by room cell, attempt 1 against later
    attempts (training horizon, every confirmation task, seeds pooled)."""
    plotted = Plotted()
    grid = representation_rows(src, "keydoor", "grid")
    shares = [abs(float(r["share"])) for r in grid if r["share"] != ""]
    limit = max(shares) if shares else 0.1
    with plt.rc_context(print_rc()):
        fig, rooms = plt.subplots(
            1, 2, figsize=(TEXT_WIDTH / 2, 1.3), layout="constrained"
        )
        image = None
        for ax, attempts, letter, title in (
            (rooms[0], "first", "a", "Attempt 1"),
            (rooms[1], "later", "b", "Attempts 2+"),
        ):
            drawn = draw_grid(ax, grid, attempts, limit)
            image = drawn if drawn is not None else image
            panel(ax, letter, title)
            cells = [r for r in grid if r["attempts"] == attempts and r["share"] != ""]
            plotted.add(
                "room",
                attempts,
                [8 * int(r["row"]) + int(r["column"]) for r in cells],
                [float(r["share"]) for r in cells],
            )
        if image is not None:
            bar = fig.colorbar(image, ax=list(rooms), shrink=0.8, pad=0.02, aspect=16)
            bar.set_label("adjusted share", fontsize=6.3, labelpad=1.5)
            bar.ax.tick_params(labelsize=5.8, length=1.5, width=0.4)
            bar.outline.set_linewidth(0.4)
    return built(
        "paper_room_attention",
        fig,
        src,
        plotted,
        {
            "figure": "position-adjusted summary share by Key-to-Door room cell (the seed's mean at the decision's segment position subtracted), attempt 1 against later attempts, training horizon, every confirmation task, seeds pooled"
        },
    )


def transplant_value(rows: Sequence[Row], name: str) -> Row | None:
    return next((r for r in rows if r["reading"] == name), None)


def figure_transplant(src: Sources) -> BuiltFigure:
    """The summary transplant: doors after the boundary, the donor's key cell,
    and CountRecall's answers against the true and the donor-consistent
    counts."""
    plotted = Plotted()
    bins = representation_rows(src, "keydoor", "transplant_bins")
    readings = representation_rows(src, "keydoor", "transplant")
    queries = representation_rows(src, "countrecall", "transplant_queries")
    styles = {
        "retained": (style_of(METHOD_CONDITION).color, "-", "o", "own summary"),
        "transplant-b8": (TRANSPLANT_BLUE, "-", "v", "donor's summary"),
        "cleared-once-b8": (
            style_of(METHOD_CONDITION).color,
            (0, (1, 1.4)),
            "x",
            "cleared once",
        ),
    }
    with plt.rc_context(print_rc()):
        fig, axes = plt.subplots(
            1,
            3,
            figsize=(TEXT_WIDTH, 1.5),
            layout="constrained",
            gridspec_kw={"width_ratios": [1.25, 0.8, 1.25]},
        )
        ax = axes[0]
        for label, (colour, linestyle, marker, _) in styles.items():
            found: dict[int, list[float]] = defaultdict(list)
            for r in bins:
                if r["label"] == label:
                    found[int(r["first_call"])].append(float(r["doors"]))
            if not found:
                continue
            xs = sorted(found)
            centres = [x + 24.5 for x in xs]
            ys = [float(np.mean(found[x])) for x in xs]
            ax.plot(
                centres,
                ys,
                color=colour,
                linestyle=linestyle,
                linewidth=1.3 if label == "retained" else 1.1,
                marker=marker,
                markersize=2.8,
                markeredgewidth=0.7,
            )
            plotted.add("keydoor bins", label, centres, ys)
        if not bins:
            pending(ax)
        mark(ax, 256.5, "boundary")
        ax.set_xlim(0, 500)
        ax.set_xticks([0, 100, 200, 300, 400, 500])
        ax.set_xlabel("Environment steps")
        ax.set_ylabel("Doors per 50 steps")
        ax.set_ylim(bottom=0)
        panel(ax, "a", "Key-to-Door")
        recessive(ax)
        ax = axes[1]
        levels = [
            ("retained", "RSM"),
            ("cleared-once-b8", "cleared"),
            ("transplant-b8", "donor"),
        ]
        drawn = False
        for slot, (label, _tick) in enumerate(levels):
            row = transplant_value(
                readings, f"T2 level, {label}: donor key visited first"
            )
            if row is None:
                continue
            estimate = float(row["estimate"])
            colour = styles[label][0]
            ax.bar(
                slot,
                estimate,
                width=0.7,
                color="white" if label == "cleared-once-b8" else colour,
                edgecolor=colour if label == "cleared-once-b8" else INK,
                linewidth=0.8 if label == "cleared-once-b8" else 0.35,
                hatch="xxxx" if label == "cleared-once-b8" else None,
                zorder=3,
            )
            ax.errorbar(
                slot,
                estimate,
                yerr=[
                    [estimate - float(row["lower"])],
                    [float(row["upper"]) - estimate],
                ],
                fmt="none",
                ecolor=INK,
                elinewidth=0.6,
                capsize=1.6,
                capthick=0.6,
                zorder=5,
            )
            plotted.add("donor key", label, [slot], [estimate])
            drawn = True
        if not drawn:
            pending(ax)
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels([tick for _, tick in levels])
        ax.tick_params(axis="x", length=0)
        ax.set_xlim(-0.6, len(levels) - 0.4)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Donor's key cell first")
        panel(ax, "b", "Next attempt")
        recessive(ax)
        ax = axes[2]
        count_styles = (
            ("retained", "true", style_of(METHOD_CONDITION).color, "-"),
            ("transplant-b2", "true", TRANSPLANT_BLUE, "-"),
            ("transplant-b2", "donor", TRANSPLANT_BLUE, (0, (3, 1.5))),
            (
                "cleared-once-b2",
                "true",
                style_of(METHOD_CONDITION).color,
                (0, (1, 1.4)),
            ),
        )
        for label, scoring, colour, linestyle in count_styles:
            found_rows = sorted(
                (r for r in queries if r["label"] == label and r["scoring"] == scoring),
                key=lambda r: int(r["query"]),
            )
            if not found_rows:
                continue
            xs = [int(r["query"]) for r in found_rows]
            ys = [float(r["accuracy"]) for r in found_rows]
            ax.plot(
                xs,
                ys,
                color=colour,
                linestyle=linestyle,
                linewidth=1.3 if label == "retained" else 1.1,
            )
            plotted.add("countrecall queries", f"{label} {scoring}", xs, ys)
        if not queries:
            pending(ax)
        mark(ax, 96.5, "rewrite", y=0.3)
        ax.set_xlim(65, 103)
        ax.set_xticks([65, 80, 96, 103])
        ax.set_ylim(0, 1.03)
        ax.set_xlabel("Query")
        ax.set_ylabel("Exact accuracy")
        panel(ax, "c", "CountRecall")
        recessive(ax)
        handles = [
            Line2D(
                [],
                [],
                color=colour,
                linestyle=linestyle,
                linewidth=1.1,
                marker=marker,
                markersize=3.0,
                label=text,
            )
            for colour, linestyle, marker, text in styles.values()
        ] + [
            Line2D(
                [],
                [],
                color=TRANSPLANT_BLUE,
                linestyle=(0, (3, 1.5)),
                linewidth=1.1,
                label="donor's summary, donor's counts",
            )
        ]
        top_legend(fig, handles, ncol=4)
    return built(
        "paper_transplant",
        fig,
        src,
        plotted,
        {
            "figure": "summary transplant at one boundary (Key-to-Door call 257, CountRecall query 65), donor = the task 128 roster places later"
        },
    )


def draw_readout(
    ax: Axes, rows: Sequence[Row], targets: Sequence[tuple[str, str]], plotted: Plotted
) -> None:
    """Ridge read-out R² per target: bars are seed means, dots the seeds."""
    drawn = False
    for slot, (target, _) in enumerate(targets):
        scores = [float(r["r2"]) for r in rows if r["target"] == target]
        if not scores:
            continue
        ax.barh(
            slot,
            float(np.mean(scores)),
            height=0.66,
            color=style_of(METHOD_CONDITION).color,
            edgecolor=INK,
            linewidth=0.35,
            zorder=3,
        )
        ax.plot(
            scores,
            [slot] * len(scores),
            marker="o",
            markersize=2.0,
            markerfacecolor="white",
            markeredgecolor=INK,
            markeredgewidth=0.45,
            linestyle="none",
            zorder=5,
        )
        plotted.add("readout", target, [slot], [float(np.mean(scores))])
        drawn = True
    if not drawn:
        pending(ax)
    ax.set_yticks(range(len(targets)))
    ax.set_yticklabels([label for _, label in targets])
    ax.invert_yaxis()
    ax.set_xlim(0, 1)
    ax.set_xlabel("Read-out R²")
    recessive(ax, "x")


def figure_representation_detail(src: Sources) -> BuiltFigure:
    """Appendix: what the summary holds. One row per benchmark: the summary one
    segment reads, projected for every seed and coloured by the hidden
    variable, beside the ridge read-out per target (Key-to-Door, CountRecall);
    and over the MazeRunner laps, the centred similarity of the summaries
    across rewrites (RSM, RSM-R) and their centred norm."""
    plotted = Plotted()
    keydoor = representation_rows(src, "keydoor", "projection")
    count = representation_rows(src, "countrecall", "projection")
    keydoor_readout = representation_rows(src, "keydoor", "readout")
    count_readout = representation_rows(src, "countrecall", "readout")
    similarity = representation_rows(src, "mazerunner", "similarity")
    norms = representation_rows(src, "mazerunner", "norms")
    seeds = ("42", "100", "2026")
    with plt.rc_context(print_rc()):
        fig = plt.figure(figsize=(TEXT_WIDTH, 4.45), layout="constrained")
        rows = fig.subfigures(3, 1, height_ratios=[1.0, 1.0, 1.08], hspace=0.02)
        for row, (
            name,
            projection,
            readout,
            colour,
            label,
            targets,
            letters,
        ) in enumerate(
            (
                (
                    "Key-to-Door",
                    keydoor,
                    keydoor_readout,
                    "key_column",
                    "key column",
                    (
                        ("key_row", "key row"),
                        ("key_column", "key column"),
                        ("door_row", "door row"),
                        ("door_column", "door column"),
                    ),
                    "abcd",
                ),
                (
                    "CountRecall",
                    count,
                    count_readout,
                    "count_0",
                    "cards of category 1",
                    (
                        ("count_0", "count 1"),
                        ("count_1", "count 2"),
                        ("count_2", "count 3"),
                        ("count_3", "count 4"),
                    ),
                    "efgh",
                ),
            )
        ):
            axes = rows[row].subplots(
                1, 4, gridspec_kw={"width_ratios": [1, 1, 1, 0.9]}
            )
            for column, seed in enumerate(seeds):
                draw_projection(
                    fig,
                    axes[column],
                    projection,
                    colour,
                    label,
                    plotted,
                    f"{name} seed {seed}",
                    seed=seed,
                )
                panel(axes[column], letters[column], f"{name}, seed {seed}")
            draw_readout(axes[3], readout, targets, plotted)
            panel(axes[3], letters[3], "Linear read-out")
        axes = rows[2].subplots(1, 3, gridspec_kw={"width_ratios": [1, 1, 1.25]})
        image = None
        for ax, cell, letter, title in (
            (axes[0], METHOD_CONDITION, "i", "MazeRunner, RSM"),
            (axes[1], ABLATION, "j", "MazeRunner, RSM-R"),
        ):
            drawn = draw_similarity(ax, similarity, cell)
            image = drawn if drawn is not None else image
            panel(ax, letter, title)
        axes[0].set_ylabel("Steps")
        if image is not None:
            bar = rows[2].colorbar(
                image, ax=list(axes[:2]), shrink=0.85, pad=0.02, aspect=18
            )
            bar.set_label("centred cosine", fontsize=6.3, labelpad=1.5)
            bar.ax.tick_params(labelsize=5.8, length=1.5, width=0.4)
            bar.outline.set_linewidth(0.4)
        ax = axes[2]
        for cell in (METHOD_CONDITION, ABLATION):
            found = sorted(
                (r for r in norms if r["cell"] == cell), key=lambda r: int(r["segment"])
            )
            if not found:
                continue
            xs = [int(r["segment"]) * SEGMENT_CALLS for r in found]
            ys = [float(r["norm"]) for r in found]
            curve(ax, xs, ys, cell, markers=False)
            plotted.add("norms", cell, xs, ys)
        if not norms:
            pending(ax)
        calls_axis(ax, horizon=False)
        ax.set_yscale("log")
        ax.set_ylabel("Centred norm (log)")
        panel(ax, "k", "Summary norm, same map")
        recessive(ax)
        ax.legend(
            handles=[
                handle(METHOD_CONDITION, markers=False),
                handle(ABLATION, markers=False),
            ],
            loc="center right",
            frameon=False,
            fontsize=6.0,
            handlelength=1.8,
        )
    return built(
        "paper_representation_detail",
        fig,
        src,
        plotted,
        {
            "figure": "what the summary holds: projections for every seed (components fitted on each seed's development captures), ridge read-out per target (bars: seed mean, dots: seeds), and over repeated MazeRunner attempts on one map the centred similarity across rewrites and the mean centred norm per segment"
        },
    )


# ------------------------------------------------------------ state, interventions

WINDOW = "raw_window"
WINDOW_COLOR = "#56B4E9"
"""Okabe-Ito sky blue for the sliding window, which ``figures_common`` does not draw."""
WINDOW_STATE_BYTES = 246_084
"""The window's per-agent state (W = 40 records per layer), tables/sliding_window.tex."""
DOSE = "figure_attention_summary_dose.csv"
KD_CLEARED = "figure2_in_context_adaptation_keydoor_summary_cleared.csv"
CR_CLEARED_BLOCKS = (
    "figure2_in_context_adaptation_countrecall_summary_cleared_blocks.csv"
)
MR_INTERVENTIONS = "figure2_in_context_adaptation_mazerunner_interventions.csv"
TMAZE_INTERVENTIONS = "figure2_in_context_adaptation_tmaze_interventions.csv"
MR_STATE = "figure_mazerunner_repeated_laps_state.csv"
MR_SAME_MAP_WINDOWS = (
    "reports/mazerunner/confirmation/laps/{history}/goals_per_window.csv"
)

STATE_NAMES: dict[str, str] = {
    "full_context": "Full history",
    "full_gru": "GRU",
    "memo": "Memo",
    WINDOW: "Sliding window",
}
"""Legend names of the per-agent state figure beside RSM (ours)."""


def colour_of(condition: str) -> str:
    if condition == WINDOW:
        return WINDOW_COLOR
    return style_of(condition).color


def state_points(src: Sources) -> dict[str, float]:
    """Peak per-agent state at 4,000 steps, as measured (bytes)."""
    peaks = {
        row["condition"]: float(row["peak_bytes"])
        for row in src.table(MR_STATE)
        if row.get("horizon") == "4000" and row.get("peak_bytes")
    }
    peaks.setdefault(WINDOW, float(WINDOW_STATE_BYTES))
    return peaks


def figure_state(src: Sources) -> BuiltFigure:
    """Deployment state against task length: the growing memories' peak state
    rises with every step, the fixed-size memories' stays flat."""
    plotted = Plotted()
    peaks = state_points(src)
    steps = np.arange(1, 4001)
    with plt.rc_context(print_rc()):
        fig, ax = plt.subplots(figsize=(TEXT_WIDTH / 2, 1.5), layout="constrained")
        drawn: list[object] = []
        for condition in ("full_context", "memo", WINDOW, METHOD_CONDITION, "full_gru"):
            peak = peaks.get(condition)
            if peak is None:
                continue
            if condition in ("full_context", "memo"):
                # One cache slot per record (full history) or four summary
                # tokens per 32-record segment (Memo): linear in the step, drawn
                # through the measured peak at 4,000 steps.
                ys = peak * steps / 4000.0
            else:
                ys = np.full_like(steps, peak, dtype=float)
            colour = colour_of(condition)
            dashed = condition == WINDOW
            ax.plot(
                steps,
                ys,
                color=colour,
                linestyle=(0, (3, 1.6)) if dashed else "-",
                linewidth=line_width(condition),
                zorder=40 if dashed else layer(condition),
            )
            # RSM (249.9 KB) and the window (246.1 KB) nearly coincide: their
            # labels sit above and below the shared line.
            nudge = {METHOD_CONDITION: 4.0, WINDOW: -4.0}.get(condition, 0.0)
            ax.annotate(
                bytes_label(peak),
                xy=(4000, peak),
                xytext=(3, nudge),
                textcoords="offset points",
                ha="left",
                va="center",
                fontsize=5.8,
                color=colour,
            )
            plotted.add(
                "state", condition, [500.0, 4000.0], [float(ys[499]), float(ys[-1])]
            )
            label = (
                "RSM (ours)"
                if condition == METHOD_CONDITION
                else STATE_NAMES[condition]
            )
            drawn.append(
                Line2D(
                    [],
                    [],
                    color=colour,
                    linewidth=line_width(condition),
                    linestyle=(0, (3, 1.6)) if dashed else "-",
                    label=label,
                )
            )
        mark(ax, 500.0, "trained")
        ax.set_yscale("log")
        ax.set_ylim(1e3, 1e8)
        ax.yaxis.set_major_locator(LogLocator(base=10, numticks=6))
        ax.yaxis.set_major_formatter(FuncFormatter(bytes_label))
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.set_xlim(0, 4000)
        ax.set_xticks([0, 1000, 2000, 3000, 4000])
        ax.set_xticklabels(["0", "1k", "2k", "3k", "4k"])
        ax.set_xlabel("Environment steps")
        ax.set_ylabel("Per-agent state")
        recessive(ax)
        if not peaks:
            pending(ax)
        fig.legend(handles=drawn, loc="outside right center", ncol=1)
    return built(
        "paper_state",
        fig,
        src,
        plotted,
        {
            "figure": "per-agent deployment state against task length: measured peaks at 4,000 steps, linear in the step for the full history and Memo, flat for RSM (RSM-R and w/o memory alike), the window and the GRU"
        },
    )


def semicolon_seeds(text: str | None) -> list[float]:
    return [
        float(part.split(":", 1)[1]) for part in (text or "").split(";") if ":" in part
    ]


def intervention_values(src: Sources) -> dict[str, dict[str, Value]]:
    """RSM with its summary kept, cleared at every boundary, unreadable or with
    its earlier buffer unreadable (read bias infinity), and w/o memory."""
    ends = {(r["environment"], r["condition"]): r for r in src.table(ENDPOINTS)}
    out: dict[str, dict[str, Value]] = defaultdict(dict)
    for environment in ("keydoor", "countrecall", "mazerunner", "tmaze"):
        for condition, key in ((METHOD_CONDITION, "kept"), (CONTROL, "w/o memory")):
            row = ends.get((environment, condition))
            value = value_of(row) if row else None
            if value is not None:
                out[environment][key] = value
    for row in src.table(KD_CLEARED):
        if row.get("condition") == METHOD_CONDITION and row.get("step") == "500":
            out["keydoor"]["cleared"] = (
                float(row["estimate"]),
                float(row["lower"]),
                float(row["upper"]),
                [],
            )
    blocks = [
        r
        for r in src.table(CR_CLEARED_BLOCKS)
        if r.get("condition") == METHOD_CONDITION
    ]
    if blocks:
        # Only the first block (queries 1-32, before the first rewrite) is
        # non-zero, so the block-weighted bounds are the interval of the total.
        sizes = [int(r["last"]) - int(r["first"]) + 1 for r in blocks]
        total = float(sum(sizes))
        weighted = [
            sum(float(r[k]) * n for r, n in zip(blocks, sizes, strict=True)) / total
            for k in ("estimate", "lower", "upper")
        ]
        per_seed = [seeds_of(r) for r in blocks]
        seeds = [
            sum(s[i] * n for s, n in zip(per_seed, sizes, strict=True)) / total
            for i in range(min(len(s) for s in per_seed))
        ]
        out["countrecall"]["cleared"] = (weighted[0], weighted[1], weighted[2], seeds)
    for table, environment in (
        (MR_INTERVENTIONS, "mazerunner"),
        (TMAZE_INTERVENTIONS, "tmaze"),
    ):
        for row in src.table(table):
            if (
                row.get("condition") == METHOD_CONDITION
                and row.get("history") == "summary-cleared"
            ):
                value = value_of(row)
                if value is not None:
                    out[environment]["cleared"] = value
    for row in src.table(DOSE):
        if number(row.get("beta")) != math.inf or row.get("target") not in (
            "summary",
            "buffer",
        ):
            continue
        environment = row["environment"]
        kept = out.get(environment, {}).get("kept")
        estimate, lower, upper = (
            number(row.get(k)) for k in ("primary", "primary_lower", "primary_upper")
        )
        if estimate is None or lower is None or upper is None:
            continue
        differences = semicolon_seeds(row.get("difference_per_seed"))
        seeds = (
            [a + b for a, b in zip(kept[3], differences, strict=False)] if kept else []
        )
        key = (
            "summary unreadable" if row["target"] == "summary" else "buffer unreadable"
        )
        out[environment][key] = (estimate, lower, upper, seeds)
    for history, key in (("retained", "kept"), ("summary-cleared", "cleared")):
        rows = src.study(
            "mazerunner", METHOD_CONDITION, MR_SAME_MAP_WINDOWS.format(history=history)
        )
        for condition, name in ((METHOD_CONDITION, key), (CONTROL, "w/o memory")):
            if history == "summary-cleared" and condition == CONTROL:
                continue
            cells = [
                r
                for r in rows
                if r.get("cell") == condition and r.get("budget") == "4000"
            ]
            last = last_window(cells)
            value = value_of(last) if last else None
            if value is not None:
                out["same map"][name] = value
    return dict(out)


INTERVENTION_BARS: tuple[tuple[str, str], ...] = (
    ("kept", "RSM, summary kept"),
    ("cleared", "summary cleared"),
    ("summary unreadable", "summary unreadable"),
    ("buffer unreadable", "earlier buffer unreadable"),
    ("w/o memory", "RSM w/o memory"),
)


def intervention_style(key: str) -> dict[str, Any]:
    method = style_of(METHOD_CONDITION).color
    return {
        "kept": {"color": method, "edgecolor": INK, "hatch": None},
        "cleared": {
            "color": "white",
            "edgecolor": SUMMARY_CLEARED.color,
            "hatch": "xxxx",
        },
        "summary unreadable": {
            "color": "white",
            "edgecolor": method,
            "hatch": "//////",
        },
        "buffer unreadable": {
            "color": "white",
            "edgecolor": BUFFER_BLUE,
            "hatch": "\\\\\\\\\\\\",
        },
        "w/o memory": {
            "color": style_of(CONTROL).color,
            "edgecolor": INK,
            "hatch": None,
        },
    }[key]


def figure_interventions(src: Sources) -> BuiltFigure:
    """What each intervention on the frozen RSM policy costs, per benchmark:
    the summary cleared at every boundary, unreadable to the decisions, or the
    earlier buffer unreadable, against the policy trained without a summary."""
    plotted = Plotted()
    values = intervention_values(src)
    panels = (
        ("keydoor", "a", "Key-to-Door", "Doors per 500 steps", False),
        ("countrecall", "b", "CountRecall", "Exact accuracy", True),
        ("mazerunner", "c", "MazeRunner", "Goal fraction", True),
        ("tmaze", "d", "T-Maze", "Success", True),
        ("same map", "e", "Same map", "Goals, last 500 of 4k", False),
    )
    transplant = {
        row["reading"]: row for row in representation_rows(src, "keydoor", "transplant")
    }
    with plt.rc_context(print_rc()):
        fig, all_axes = plt.subplots(
            1,
            6,
            figsize=(TEXT_WIDTH, 1.5),
            layout="constrained",
            gridspec_kw={"width_ratios": [5, 5, 3, 3, 3.3, 3.9]},
        )
        axes = all_axes[:5]
        for ax, (environment, letter, title, ylabel, fraction) in zip(
            axes, panels, strict=True
        ):
            found = [
                (k, values.get(environment, {}).get(k)) for k, _ in INTERVENTION_BARS
            ]
            bars = [(k, v) for k, v in found if v is not None]
            highest = max((max([v[2], *v[3]]) for _, v in bars), default=1.0)
            for slot, (key, (estimate, lower, upper, seeds)) in enumerate(bars):
                st = intervention_style(key)
                ax.bar(
                    slot,
                    estimate,
                    width=0.72,
                    color=st["color"],
                    edgecolor=st["edgecolor"],
                    linewidth=0.6,
                    hatch=st["hatch"],
                    zorder=3,
                )
                ax.errorbar(
                    slot,
                    estimate,
                    yerr=[[estimate - lower], [upper - estimate]],
                    fmt="none",
                    ecolor=INK,
                    elinewidth=0.6,
                    capsize=1.4,
                    capthick=0.6,
                    zorder=5,
                )
                for offset, seed in zip((-0.16, 0.0, 0.16), seeds, strict=False):
                    ax.plot(
                        slot + offset,
                        seed,
                        marker="o",
                        markersize=1.8,
                        markerfacecolor="white",
                        markeredgecolor=INK,
                        markeredgewidth=0.4,
                        linestyle="none",
                        zorder=6,
                    )
                plotted.add(letter, key, [slot], [estimate], [lower], [upper])
            if not bars:
                pending(ax)
            ax.set_xticks([])
            ax.set_xlim(-0.6, max(len(bars), 1) - 0.4)
            ax.set_ylim(0, 1.05 if fraction else highest * 1.12)
            if fraction:
                ax.set_yticks([0, 0.5, 1.0])
                ax.set_yticklabels(["0", "0.5", "1"])
            else:
                ax.yaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
            ax.set_ylabel(ylabel)
            panel(ax, letter, title)
            recessive(ax)
        # (f) The transplant: how often the next Key-to-Door attempt visits
        # the donor task's key cell first.
        ax = all_axes[5]
        method = style_of(METHOD_CONDITION).color
        kinds = (
            ("retained", {"color": method, "edgecolor": INK, "hatch": None}),
            (
                "transplant-b8",
                {"color": TRANSPLANT_BLUE, "edgecolor": INK, "hatch": None},
            ),
            (
                "cleared-once-b8",
                {"color": "white", "edgecolor": SUMMARY_CLEARED.color, "hatch": "xxxx"},
            ),
        )
        for slot, (history, st) in enumerate(kinds):
            row = transplant.get(f"T2 level, {history}: donor key visited first")
            if row is None:
                continue
            estimate, lower, upper = (
                float(row[k]) for k in ("estimate", "lower", "upper")
            )
            ax.bar(
                slot,
                estimate,
                width=0.72,
                color=st["color"],
                edgecolor=st["edgecolor"],
                linewidth=0.6,
                hatch=st["hatch"],
                zorder=3,
            )
            ax.errorbar(
                slot,
                estimate,
                yerr=[[estimate - lower], [upper - estimate]],
                fmt="none",
                ecolor=INK,
                elinewidth=0.6,
                capsize=1.4,
                capthick=0.6,
                zorder=5,
            )
            for offset, seed in zip(
                (-0.16, 0.0, 0.16), semicolon_seeds(row.get("per_seed")), strict=False
            ):
                ax.plot(
                    slot + offset,
                    seed,
                    marker="o",
                    markersize=1.8,
                    markerfacecolor="white",
                    markeredgecolor=INK,
                    markeredgewidth=0.4,
                    linestyle="none",
                    zorder=6,
                )
            plotted.add("f", history, [slot], [estimate], [lower], [upper])
        if not transplant:
            pending(ax)
        # The legend names the three bar styles (own summary, donor, cleared once).
        ax.set_xticks([])
        ax.set_xlim(-0.6, len(kinds) - 0.4)
        ax.set_ylim(0, 1.05)
        ax.set_yticks([0, 0.5, 1.0])
        ax.set_yticklabels(["0", "0.5", "1"])
        ax.set_ylabel("Donor key first")
        panel(ax, "f", "Transplant")
        recessive(ax)
        handles = [
            Patch(
                facecolor=intervention_style(k)["color"],
                edgecolor=intervention_style(k)["edgecolor"],
                hatch=intervention_style(k)["hatch"],
                linewidth=0.6,
                label=label,
            )
            for k, label in INTERVENTION_BARS
        ] + [
            Patch(
                facecolor=TRANSPLANT_BLUE,
                edgecolor=INK,
                linewidth=0.6,
                label="donor summary",
            )
        ]
        top_legend(fig, handles, ncol=6)
    return built(
        "paper_interventions",
        fig,
        src,
        plotted,
        {
            "figure": "interventions on the frozen RSM policy: summary cleared at every boundary, summary or earlier buffer unreadable (read bias infinity; Key-to-Door and CountRecall), against the policy trained without a summary; bars: means, whiskers: 95 % joint intervals, dots: seeds where saved"
        },
    )


def figure_retention_age(src: Sources) -> BuiltFigure:
    """CountRecall accuracy by the age of the queried evidence (summary
    rewrites since it was dealt), every compared memory; one panel."""
    plotted = Plotted()
    retention = by_condition(src.table(CR_RETENTION), history="retained")
    with plt.rc_context(print_rc()):
        fig, ax = plt.subplots(figsize=(TEXT_WIDTH / 2, 1.3), layout="constrained")
        offsets = dict(
            zip(COMPARED, np.linspace(-0.28, 0.28, len(COMPARED)), strict=True)
        )
        for c in (*COMPARED[1:], COMPARED[0]):
            rows = {int(r["stratum"]): r for r in retention.get(c, [])}
            if not rows:
                continue
            strata = sorted(rows)
            xs = [s + offsets[c] for s in strata]
            ys = [float(rows[s]["estimate"]) for s in strata]
            lo = [float(rows[s]["lower"]) for s in strata]
            hi = [float(rows[s]["upper"]) for s in strata]
            st = style_of(c)
            ax.errorbar(
                xs,
                ys,
                yerr=[np.subtract(ys, lo), np.subtract(hi, ys)],
                color=st.color,
                linestyle=st.linestyle,
                linewidth=line_width(c) * (1.0 if c == METHOD_CONDITION else 0.6),
                marker=st.marker,
                markersize=3.2 if c == METHOD_CONDITION else 2.8,
                markerfacecolor="white" if st.open_marker else st.color,
                markeredgecolor=st.color,
                markeredgewidth=0.6,
                elinewidth=0.8,
                capsize=1.6,
                capthick=0.6,
                zorder=layer(c),
            )
            plotted.add("retention", c, strata, ys, lo, hi)
        if not retention:
            pending(ax)
        ax.set_xticks(range(4))
        ax.set_xticklabels(["in buffer", "1", "2", "3"])
        ax.set_xlim(-0.45, 3.45)
        for boundary in (0.5, 1.5, 2.5):
            ax.axvline(boundary, color=GRID, linewidth=0.5, zorder=0)
        lowest = min(
            (float(r["lower"]) for c in COMPARED for r in retention.get(c, [])),
            default=0.6,
        )
        ax.set_ylim(min(0.6, math.floor(lowest * 20) / 20), 1.02)
        ax.set_xlabel("Rewrites since the evidence was dealt")
        ax.set_ylabel("Exact accuracy")
        recessive(ax)
        fig.legend(
            handles=[
                handle(c, markers=True, label=BAR_TICKS[c].replace("\n", " "))
                for c in COMPARED
            ],
            loc="outside right center",
            ncol=1,
        )
    return built(
        "paper_retention_age",
        fig,
        src,
        plotted,
        {
            "figure": "CountRecall accuracy by evidence age (summary rewrites since the queried category was first dealt), every compared memory, 95 % joint intervals"
        },
    )


PAPER_FIGURES = (
    figure_results_at_a_glance,
    figure_in_context,
    figure_countrecall,
    figure_attention,
    figure_transplant,
    figure_representation_detail,
    figure_ablations,
    figure_new_tasks,
    figure_learning,
    figure_state,
    figure_interventions,
    figure_retention_age,
    figure_room_attention,
)
"""Every paper figure in the order of the paper; the first is the one the
headless test checks."""


def build_paper_figures(data: Path, layout: StudyLayout) -> list[BuiltFigure]:
    """Draw every figure from the saved tables (``data``) and study roots."""
    return [make(Sources(data, layout)) for make in PAPER_FIGURES]

In [ ]:
# The print style for this kernel: Matplotlib resolves fonts and the PDF font
# type when a figure is drawn and saved, not when it is built.
plt.rcParams.update(print_rc())
figures = build_paper_figures(DATA_DIR, layout)
for built_figure in figures:
    display(built_figure.figure)

In [ ]:
# What each figure read, and what is missing.
for built_figure in figures:
    missing = built_figure.notes.get("missing") or "nothing"
    print(
        f"{built_figure.name}: {len(set(built_figure.sources))} files read; missing {missing}"
    )

In [ ]:
# Save: each figure (PDF + PNG), the plotted values and the notes.
for built_figure in figures:
    for path in built_figure.save(FIGURES_DIR, DATA_DIR, panel_formats=()):
        print(path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    built_figure.close()